In [2]:
import pandas as pd
import numpy as np

#load
customer=pd.read_csv('Data Generation/KartZone_Customers.csv')

#Data Undestanding

print("No of rows    :",customer.shape[0])
print("No of Columns :",customer.shape[1])
print()

print(customer.head())
print()

print(customer.tail())
print()

print(customer.info())
print()

print(customer.describe())
print()

print(customer.columns.tolist())


No of rows    : 1035
No of Columns : 17

  Customer_ID   Customer_Name   Age  Gender       City        State Region  \
0       C1001   Pallavi Gupta  36.0    Male   chennai    Tamil Nadu  South   
1       C1002    Shreya Reddy -10.0       F     MUMBAI  maharashtra   West   
2       C1003     PALLAVI DAS  32.0    Male        Hyd    Telangana  South   
3       C1004    Shreya Patel  40.0       f  Hyderabad          NaN  south   
4       C1005     Aarav Dubey  35.0  FEMALE     mumbai  Maharashtra   West   

  Customer_Segment Registration_Date Registration_Source Last_Login_Date  \
0          regular        14/05/2022                 app      07/03/2023   
1              New        09/25/2024                 NaN      05-06-2024   
2              New        05-08-2023        instagram ad             NaN   
3         Newcomer        28-04-2023                 APP      01/12/2024   
4          PREMIUM        21-03-2020             Website             NaN   

  Last_Order_Date                

In [3]:
#Remove Duplicates and Blank Rows

In [4]:
print(f"Before : {len(customer)}")
print()

#Remove full balnk rows
customer=customer.dropna(how='all')

#Remove duplicates
customer=customer.drop_duplicates()

#Reset_index
customer=customer.reset_index(drop=True)
print()

print(f"After: {len(customer)}")

Before : 1035


After: 1000


In [5]:
#Clean Customer Name
#REmove extra spaces and fix case

customer['Customer_Name']=customer['Customer_Name'].astype(str).str.strip().str.replace(r'\s+',' ',regex=True).str.title().str.replace(r'^(MR\.|MRS\.|MS\.) ','',regex=True)

customer['Customer_Name']=customer['Customer_Name'].replace(['Nan','nan','NaN','None','NA','N/A',''],np.nan)

print(customer[['Customer_Name','Phone']].value_counts().head())

Customer_Name    Phone     
Mohan Patel      0000000000    2
Sonal Iyer       XXXXXXXXXX    2
Ishaan Malhotra  123           2
Rohit Agarwal    123           1
Riya Gupta       9302228735    1
Name: count, dtype: int64


In [6]:
#Clean and Validate Age

customer['Age']=pd.to_numeric(customer['Age'],errors='coerce')
print(customer['Age'].describe())
print()

print(f"Invalid ages (< 18): {(customer['Age'] < 18).sum()}")
print(f"Invalid ages (> 80): {(customer['Age'] > 80).sum()}")

customer['Age']=customer['Age'].where((customer['Age']>=18) &(customer['Age']<=80),np.nan)

# Fill missing ages with median
median_age = customer['Age'].median()
customer['Age'] = customer['Age'].fillna(median_age)

customer['Age'] = customer['Age'].astype(int)
print()
print(customer['Age'].describe())

count    940.000000
mean      33.685106
std       32.227275
min      -20.000000
25%       22.000000
50%       31.000000
75%       39.000000
max      199.000000
Name: Age, dtype: float64

Invalid ages (< 18): 113
Invalid ages (> 80): 41

count    1000.000000
mean       32.323000
std         8.362332
min        18.000000
25%        27.000000
50%        32.000000
75%        37.000000
max        65.000000
Name: Age, dtype: float64


In [7]:
#Standardize Gender

In [8]:
print(customer['Gender'].value_counts())

def standardize_gender(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    if val in ['male','m','1','man']:
        return 'Male'
    elif val in ['female','f','2','woman']:
        return 'Female'
    return np.nan

customer['Gender'] = customer['Gender'].apply(standardize_gender)

# Fill remaining nulls with mode
customer['Gender'] = customer['Gender'].fillna(
    customer['Gender'].mode()[0]
)

print(customer['Gender'].value_counts())
print()

print(customer['Gender'].value_counts())


Gender
F         103
2          96
M          89
Male       86
Female     86
female     83
f          82
male       81
1          81
m          80
FEMALE     70
MALE       63
Name: count, dtype: int64
Gender
Female    520
Male      480
Name: count, dtype: int64

Gender
Female    520
Male      480
Name: count, dtype: int64


In [9]:
#STEP 7 — Standardize City
def standardize_city(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    city_map = {
        'mumbai':'Mumbai', 'bombay':'Mumbai',
        'delhi':'Delhi', 'new delhi':'Delhi',
        'bangalore':'Bangalore', 'bengaluru':'Bangalore', 'banglore':'Bangalore',
        'chennai':'Chennai', 'madras':'Chennai',
        'hyderabad':'Hyderabad', 'hyd':'Hyderabad', 'hydrabad':'Hyderabad',
        'pune':'Pune', 'poona':'Pune'
    }
    for key, standard in city_map.items():
        if key in val:
            return standard
    return np.nan

customer['City'] = customer['City'].apply(standardize_city)

# Fill missing with mode
customer['City'] = customer['City'].fillna(
    customer['City'].mode()[0]
)

print(customer['City'].value_counts())

City
Mumbai       283
Delhi        205
Bangalore    205
Chennai      126
Hyderabad    112
Pune          69
Name: count, dtype: int64


In [10]:
# Re-derive State and Region from clean City
city_state_map = {
    'Mumbai':    ('Maharashtra', 'West'),
    'Delhi':     ('Delhi', 'North'),
    'Bangalore': ('Karnataka', 'South'),
    'Chennai':   ('Tamil Nadu', 'South'),
    'Hyderabad': ('Telangana', 'South'),
    'Pune':      ('Maharashtra', 'West')
}

customer['State']  = customer['City'].map(
    {k: v[0] for k, v in city_state_map.items()}
)
customer['Region'] = customer['City'].map(
    {k: v[1] for k, v in city_state_map.items()}
)

print(customer[['City','State','Region']].value_counts())

City       State        Region
Mumbai     Maharashtra  West      283
Bangalore  Karnataka    South     205
Delhi      Delhi        North     205
Chennai    Tamil Nadu   South     126
Hyderabad  Telangana    South     112
Pune       Maharashtra  West       69
Name: count, dtype: int64


In [11]:
def standardize_segment(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    if val in ['premium','prem','vip']:
        return 'Premium'
    elif val in ['regular','reg','standard']:
        return 'Regular'
    elif val in ['new','newcomer','first time']:
        return 'New'
    elif val in ['at-risk','atrisk','at risk']:
        return 'At-Risk'
    elif val in ['churned','inactive','lost']:
        return 'Churned'
    return np.nan

customer['Customer_Segment'] = customer['Customer_Segment'].apply(
    standardize_segment
)

# Fill nulls with mode
customer['Customer_Segment'] = customer['Customer_Segment'].fillna(
    customer['Customer_Segment'].mode()[0]
)

print(customer['Customer_Segment'].value_counts())

Customer_Segment
Regular    379
New        284
At-Risk    127
Premium    121
Churned     89
Name: count, dtype: int64


In [12]:
def standardize_source(val):
    if pd.isnull(val):
        return 'Unknown'
    val = str(val).strip().lower()
    if 'app' in val or 'mobile' in val:
        return 'App'
    elif 'web' in val:
        return 'Website'
    elif 'refer' in val:
        return 'Referral'
    elif 'insta' in val:
        return 'Instagram Ad'
    elif 'google' in val:
        return 'Google Ad'
    elif 'organic' in val or 'direct' in val:
        return 'Organic'
    return 'Unknown'

customer['Registration_Source'] = customer['Registration_Source'].apply(
    standardize_source
)

print(customer['Registration_Source'].value_counts())

Registration_Source
App             267
Website         184
Instagram Ad    161
Referral        131
Google Ad       110
Organic          82
Unknown          65
Name: count, dtype: int64


In [13]:
# Parse All Date Columns
from datetime import datetime

def parse_date(val):
    if pd.isnull(val) or str(val).strip() in ['','NA','N/A','nan','null','-']:
        return np.nan
    val = str(val).strip()
    formats = ['%Y-%m-%d','%d/%m/%Y','%d-%m-%Y',
               '%m/%d/%Y','%d.%m.%Y','%Y/%m/%d']
    for fmt in formats:
        try:
            return datetime.strptime(val, fmt).strftime('%Y-%m-%d')
        except:
            continue
    return np.nan

# Apply to all date columns
date_cols = ['Registration_Date','Last_Login_Date','Last_Order_Date']
for col in date_cols:
    customer[col] = customer[col].apply(parse_date)
    customer[col] = pd.to_datetime(customer[col], errors='coerce')

print(customer[date_cols].dtypes)
print()
print(customer[date_cols].isnull().sum())

Registration_Date    datetime64[ns]
Last_Login_Date      datetime64[ns]
Last_Order_Date      datetime64[ns]
dtype: object

Registration_Date      0
Last_Login_Date      194
Last_Order_Date      220
dtype: int64


In [14]:
#Validate and Clean Phone
import re

def clean_phone(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip()
    # Remove country codes and special characters
    val = re.sub(r'[\+\-\s\(\)]', '', val)
    val = re.sub(r'^(91|0091)', '', val)
    # Must be 10 digits starting with 6-9
    if re.match(r'^[6-9]\d{9}$', val):
        return val
    return np.nan

customer['Phone'] = customer['Phone'].apply(clean_phone)

valid_pct = customer['Phone'].notna().sum() / len(customer) * 100
print(f"Valid phones: {valid_pct:.1f}%")

Valid phones: 63.6%


In [15]:
#Validate and Clean Email

def clean_email(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    if val in ['','na','n/a','nan','null','-']:
        return np.nan
    # Basic email validation
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    if re.match(pattern, val):
        return val
    return np.nan

customer['Email'] = customer['Email'].apply(clean_email)

valid_pct = customer['Email'].notna().sum() / len(customer) * 100
print(f"Valid emails: {valid_pct:.1f}%")

Valid emails: 58.5%


In [16]:
# Clean Loyalty_Score

# Convert to numeric
customer['Loyalty_Score'] = pd.to_numeric(
    customer['Loyalty_Score'], errors='coerce'
)

# Valid range 1-100
customer['Loyalty_Score'] = customer['Loyalty_Score'].where(
    (customer['Loyalty_Score'] >= 1) & (customer['Loyalty_Score'] <= 100),
    np.nan
)

# Fill nulls with segment median
customer['Loyalty_Score'] = customer.groupby('Customer_Segment')['Loyalty_Score'].transform(
    lambda x: x.fillna(x.median())
)

print(customer.groupby('Customer_Segment')['Loyalty_Score'].describe())

                  count       mean        std   min   25%   50%   75%    max
Customer_Segment                                                            
At-Risk           127.0  27.377953  12.532453   6.0  17.0  28.0  37.0   50.0
Churned            89.0  15.769663   9.223259   1.0   8.0  15.5  22.0   35.0
New               284.0  33.422535  12.099601  10.0  23.0  34.0  42.0   55.0
Premium           121.0  75.842975  10.659900  57.0  67.0  75.0  82.0  100.0
Regular           379.0  56.134565  15.550578  30.0  42.0  56.0  68.5   85.0


In [17]:
#Standardize Newsletter_Subscribed
def standardize_bool(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    if val in ['yes','1','true','y']:
        return 'Yes'
    elif val in ['no','0','false','n']:
        return 'No'
    return np.nan

customer['Newsletter_Subscribed'] = customer['Newsletter_Subscribed'].apply(
    standardize_bool
)
customer['Newsletter_Subscribed'] = customer['Newsletter_Subscribed'].fillna('No')

print(customer['Newsletter_Subscribed'].value_counts())

Newsletter_Subscribed
No     546
Yes    454
Name: count, dtype: int64


In [18]:
# Standardize Preferred_Category

def standardize_category(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    if 'electron' in val:     return 'Electronics'
    if 'fashion' in val or 'cloth' in val: return 'Fashion'
    if 'home' in val or 'kitchen' in val:  return 'Home & Kitchen'
    if 'beauty' in val or 'personal' in val: return 'Beauty'
    return np.nan

customer['Preferred_Category'] = customer['Preferred_Category'].apply(
    standardize_category
)
customer['Preferred_Category'] = customer['Preferred_Category'].fillna('Unknown')

print(customer['Preferred_Category'].value_counts())

Preferred_Category
Electronics       267
Fashion           223
Home & Kitchen    190
Unknown           177
Beauty            143
Name: count, dtype: int64


In [19]:
#Outlier Treatment on Loyalty_Score

# IQR method
Q1  = customer['Loyalty_Score'].quantile(0.25)
Q3  = customer['Loyalty_Score'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Loyalty Score bounds: {lower:.1f} to {upper:.1f}")

outliers = ((customer['Loyalty_Score'] < lower) |
            (customer['Loyalty_Score'] > upper)).sum()
print(f"Outliers found: {outliers}")

# Cap outliers
customer['Loyalty_Score'] = customer['Loyalty_Score'].clip(
    lower=max(1,lower), upper=min(100,upper)
)

Loyalty Score bounds: -19.0 to 109.0
Outliers found: 0


In [20]:
#Feature Engineering

today = pd.Timestamp.now()

# 1. Customer Tenure
customer['Customer_Tenure_Days'] = (
    today - customer['Registration_Date']
).dt.days

# 2. Days Since Last Login
customer['Days_Since_Login'] = (
    today - customer['Last_Login_Date']
).dt.days

# 3. Days Since Last Order
customer['Days_Since_Order'] = (
    today - customer['Last_Order_Date']
).dt.days

# 4. Age Group
def age_group(age):
    if pd.isnull(age): return 'Unknown'
    if age <= 25:   return '18-25'
    elif age <= 35: return '26-35'
    elif age <= 45: return '36-45'
    else:           return '46+'

customer['Age_Group'] = customer['Age'].apply(age_group)

# 5. Churn Flag
customer['Churn_Flag'] = customer['Customer_Segment'].isin(
    ['Churned','At-Risk']
).astype(int)

# 6. Is Dormant — no login in 90 days
customer['Is_Dormant'] = (
    customer['Days_Since_Login'] > 90
).astype(int)

# 7. Is Valid Contact
customer['Is_Valid_Contact'] = (
    customer['Phone'].notna() & customer['Email'].notna()
).astype(int)

# 8. Tenure Band
def tenure_band(days):
    if pd.isnull(days): return 'Unknown'
    if days < 90:    return 'New (< 3 months)'
    elif days < 365: return 'Growing (3-12 months)'
    elif days < 730: return 'Established (1-2 years)'
    else:            return 'Loyal (2+ years)'

customer['Tenure_Band'] = customer['Customer_Tenure_Days'].apply(tenure_band)

print(customer[['Age_Group','Churn_Flag','Is_Dormant',
                'Is_Valid_Contact','Tenure_Band']].head(10))

  Age_Group  Churn_Flag  Is_Dormant  Is_Valid_Contact              Tenure_Band
0     36-45           0           1                 1         Loyal (2+ years)
1     26-35           0           1                 1  Established (1-2 years)
2     26-35           0           0                 0         Loyal (2+ years)
3     36-45           0           1                 1         Loyal (2+ years)
4     26-35           0           0                 1         Loyal (2+ years)
5     26-35           0           1                 1         Loyal (2+ years)
6       46+           0           1                 0  Established (1-2 years)
7     26-35           0           1                 1  Established (1-2 years)
8     26-35           0           0                 1         Loyal (2+ years)
9     36-45           0           0                 0         Loyal (2+ years)


In [21]:
# Normalization
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Min-Max Normalization — Age and Loyalty_Score
scaler_mm = MinMaxScaler()
customer['Age_Normalized'] = scaler_mm.fit_transform(
    customer[['Age']]
)
customer['Loyalty_Normalized'] = scaler_mm.fit_transform(
    customer[['Loyalty_Score']]
)

# Z-Score Standardization — Tenure
scaler_z = StandardScaler()
customer['Tenure_Zscore'] = scaler_z.fit_transform(
    customer[['Customer_Tenure_Days']].fillna(0)
)

# Log Transformation — Days Since Login (right skewed)
customer['Days_Since_Login_Log'] = np.log1p(
    customer['Days_Since_Login'].fillna(0)
)

print(customer[['Age_Normalized','Loyalty_Normalized',
                'Tenure_Zscore','Days_Since_Login_Log']].describe())

       Age_Normalized  Loyalty_Normalized  Tenure_Zscore  Days_Since_Login_Log
count     1000.000000         1000.000000   1.000000e+03           1000.000000
mean         0.304745            0.442672  -3.730349e-17              5.404198
std          0.177922            0.223562   1.000500e+00              2.663946
min          0.000000            0.000000  -1.678129e+00              0.000000
25%          0.191489            0.282828  -8.851450e-01              6.281330
50%          0.297872            0.414141   4.606584e-02              6.620073
75%          0.404255            0.606061   9.036165e-01              6.863019
max          1.000000            1.000000   1.633853e+00              7.365180


In [22]:
# Encoding Categorical Variables

# 1. Label Encoding — Segment (ordinal)
segment_order = {'Churned':0,'At-Risk':1,'New':2,'Regular':3,'Premium':4}
customer['Segment_Encoded'] = customer['Customer_Segment'].map(segment_order)

# 2. Binary Encoding — Gender
customer['Gender_Encoded'] = customer['Gender'].map({'Male':1,'Female':0})

# 3. One Hot Encoding — Region
region_dummies = pd.get_dummies(
    customer['Region'],
    prefix='Region',
    drop_first=True
)
customer = pd.concat([customer, region_dummies], axis=1)

# 4. One Hot Encoding — Preferred Category
cat_dummies = pd.get_dummies(
    customer['Preferred_Category'],
    prefix='PrefCat',
    drop_first=True
)
customer = pd.concat([customer, cat_dummies], axis=1)

print(f"Shape after encoding: {customer.shape}")
print(customer[['Segment_Encoded','Gender_Encoded']].head())

Shape after encoding: (1000, 37)
   Segment_Encoded  Gender_Encoded
0                3               1
1                2               0
2                2               1
3                2               0
4                4               0


In [23]:
#Final Validation

print("\n" + "="*55)
print("  FINAL DATA QUALITY REPORT")
print("="*55)
print(f"  Total rows        : {len(customer):,}")
print(f"  Total columns     : {len(customer.columns)}")
print(f"  Remaining nulls   : {customer.isnull().sum().sum():,}")
print(f"  Duplicates        : {customer.duplicated().sum()}")
print(f"\n  NULLS PER COLUMN:")
for col in customer.columns:
    n = customer[col].isnull().sum()
    if n > 0:
        print(f"    {col:<35} {n:>4} ({n/len(customer)*100:.1f}%)")

print(f"\n  SEGMENT DISTRIBUTION:")
print(customer['Customer_Segment'].value_counts())
print(f"\n  CITY DISTRIBUTION:")
print(customer['City'].value_counts())
print(f"\n  AGE STATS:")
print(customer['Age'].describe())


  FINAL DATA QUALITY REPORT
  Total rows        : 1,000
  Total columns     : 37
  Remaining nulls   : 1,607
  Duplicates        : 0

  NULLS PER COLUMN:
    Last_Login_Date                      194 (19.4%)
    Last_Order_Date                      220 (22.0%)
    Email                                415 (41.5%)
    Phone                                364 (36.4%)
    Days_Since_Login                     194 (19.4%)
    Days_Since_Order                     220 (22.0%)

  SEGMENT DISTRIBUTION:
Customer_Segment
Regular    379
New        284
At-Risk    127
Premium    121
Churned     89
Name: count, dtype: int64

  CITY DISTRIBUTION:
City
Mumbai       283
Delhi        205
Bangalore    205
Chennai      126
Hyderabad    112
Pune          69
Name: count, dtype: int64

  AGE STATS:
count    1000.000000
mean       32.323000
std         8.362332
min        18.000000
25%        27.000000
50%        32.000000
75%        37.000000
max        65.000000
Name: Age, dtype: float64


In [24]:
#Save Clean Data

# Save cleaned version
customer.to_csv('KartZone_Customers_Clean.csv', index=False)
print("Saved → KartZone_Customers_Clean.csv ✓")

# Save only essential columns for SQL load
essential_cols = [
    'Customer_ID','Customer_Name','Age','Gender','City','State','Region',
    'Customer_Segment','Registration_Date','Registration_Source',
    'Last_Login_Date','Last_Order_Date','Email','Phone','Loyalty_Score',
    'Preferred_Category','Newsletter_Subscribed',
    'Customer_Tenure_Days','Days_Since_Login','Days_Since_Order',
    'Age_Group','Churn_Flag','Is_Dormant','Is_Valid_Contact','Tenure_Band'
]
customer[essential_cols].to_csv('KartZone_Customers_Final.csv', index=False)
print("Saved → KartZone_Customers_Final.csv ✓")


Saved → KartZone_Customers_Clean.csv ✓
Saved → KartZone_Customers_Final.csv ✓


In [25]:
#Products Data — Complete Cleaning Pipeline

In [26]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load
products = pd.read_csv('Data Generation/KartZone_Products.csv')

# First look
print(products.shape)
print(products.head())
print(products.info())
print(products.describe())
print(products.columns.tolist())

(825, 14)
  Product_ID              Product_Name        Category Sub_Category  \
0  ELEC-1000          Sony WiFi Router     electronics  WiFi Router   
1  HOME-3000       Story@Home LED Lamp  HOME & KITCHEN     LED Lamp   
2  BEAU-4000            Dove Sunscreen          Beauty    Sunscreen   
3  ELEC-1001            Sony Pen Drive      Electronis    Pen Drive   
4  FASH-2000  Fabindia T-Shirt XL Grey          Fasion      T-Shirt   

        Brand         MRP Cost_Price Discount_Pct Selling_Price Launch_Date  \
0        Sony    72077.91        NaN            0      72077.91  10/11/2024   
1  Story@Home  Rs.1257.04     723.91         10.0       1131.34  09/26/2024   
2        Dove         NaN    1626.91           10      ₹3223.38  03/01/2022   
3        Sony    81550.83   60218.98          NaN      73395.75  11/03/2023   
4    Fabindia     2193.72    2331.12           30        1535.6  16-08-2020   

   Rating  Stock_Units Weight_KG Product_Status  
0     3.3        427.0      0.39   Dis

In [39]:
#Understand Data Quality

# Missing values
print(products.isnull().sum())
print(products.isnull().sum() / len(products) * 100)

# Duplicates
print(f"Duplicate rows: {products.duplicated().sum()}")

# Blank rows
print(f"Blank rows: {products.isnull().all(axis=1).sum()}")

# Unique values per column
print(products.nunique())

# Sample messy values per column
for col in products.columns:
    print(f"\n{col}:")
    print(products[col].value_counts(dropna=False).head(8))

Product_ID         10
Product_Name       10
Category           10
Sub_Category       10
Brand              10
MRP                70
Cost_Price         68
Discount_Pct       91
Selling_Price      94
Launch_Date       167
Rating             89
Stock_Units        86
Weight_KG          92
Product_Status    119
dtype: int64
Product_ID         1.212121
Product_Name       1.212121
Category           1.212121
Sub_Category       1.212121
Brand              1.212121
MRP                8.484848
Cost_Price         8.242424
Discount_Pct      11.030303
Selling_Price     11.393939
Launch_Date       20.242424
Rating            10.787879
Stock_Units       10.424242
Weight_KG         11.151515
Product_Status    14.424242
dtype: float64
Duplicate rows: 24
Blank rows: 10
Product_ID        800
Product_Name      564
Category           20
Sub_Category       60
Brand              60
MRP               741
Cost_Price        742
Discount_Pct       28
Selling_Price     717
Launch_Date       609
Rating            

In [41]:
# Remove Duplicates and Blank Rows
print(f"Before: {len(products)}")

# Remove blank rows first
products = products.dropna(how='all')

# Remove exact duplicates
products = products.drop_duplicates()

# Remove duplicate Product_IDs — keep first
products = products.drop_duplicates(subset=['Product_ID'], keep='first')

# Reset index
products = products.reset_index(drop=True)

print(f"After: {len(products)}")

Before: 825
After: 800


In [42]:
#Clean Product_ID

# Check format
print(products['Product_ID'].value_counts().head(10))

# Strip whitespace
products['Product_ID'] = products['Product_ID'].astype(str).str.strip()

# Check for nulls or INVALID entries
invalid_ids = products['Product_ID'].str.contains('INVALID', na=False)
print(f"Invalid Product IDs: {invalid_ids.sum()}")

# Remove invalid product rows
products = products[~invalid_ids].reset_index(drop=True)

print(f"Products after removing invalid IDs: {len(products)}")

Product_ID
ELEC-1000    1
HOME-3098    1
ELEC-1182    1
ELEC-1183    1
FASH-2171    1
FASH-2172    1
FASH-2173    1
FASH-2174    1
HOME-3097    1
ELEC-1184    1
Name: count, dtype: int64
Invalid Product IDs: 0
Products after removing invalid IDs: 800


In [43]:
# Clean Product_Name

# Strip whitespace and fix case
products['Product_Name'] = (
    products['Product_Name']
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

# Replace missing encodings
products['Product_Name'] = products['Product_Name'].replace(
    ['nan','NaN','NA','N/A','None','','null','-'], np.nan
)

# Fill missing with Category + Sub_Category
products['Product_Name'] = products.apply(
    lambda row: f"{row['Brand']} {row['Sub_Category']}"
    if pd.isnull(row['Product_Name']) else row['Product_Name'],
    axis=1
)

print(products['Product_Name'].head(10))

0                Sony WiFi Router
1             Story@Home LED Lamp
2                  Dove Sunscreen
3                  Sony Pen Drive
4        Fabindia T-Shirt XL Grey
5    Xiaomi Earphones Wired Black
6                Pigeon Knife Set
7                    Apple Camera
8              Fire-Boltt Printer
9             Solimo Air Purifier
Name: Product_Name, dtype: object


In [44]:
#Standardize Category

def standardize_category(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    if 'electron' in val:                    return 'Electronics'
    if 'fashion' in val or 'cloth' in val:   return 'Fashion'
    if 'home' in val or 'kitchen' in val:    return 'Home & Kitchen'
    if 'beauty' in val or 'personal' in val: return 'Beauty'
    return np.nan

products['Category'] = products['Category'].apply(standardize_category)

# Fill nulls — derive from Product_ID prefix
def derive_cat_from_id(pid):
    if pd.isnull(pid): return np.nan
    pid = str(pid).upper()
    if pid.startswith('ELEC'): return 'Electronics'
    if pid.startswith('FASH'): return 'Fashion'
    if pid.startswith('HOME'): return 'Home & Kitchen'
    if pid.startswith('BEAU'): return 'Beauty'
    return np.nan

products['Category'] = products.apply(
    lambda row: derive_cat_from_id(row['Product_ID'])
    if pd.isnull(row['Category']) else row['Category'],
    axis=1
)

print(products['Category'].value_counts())

Category
Electronics       275
Fashion           259
Home & Kitchen    152
Beauty            114
Name: count, dtype: int64


In [45]:
# Clean MRP

def clean_price(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip()
    # Remove currency symbols and units
    val = re.sub(r'[₹Rs\.INR,/\-\s]', '', val)
    try:
        result = float(val)
        return result if result > 0 else np.nan
    except:
        return np.nan

products['MRP'] = products['MRP'].apply(clean_price)

# Check distribution
print(products['MRP'].describe())
print(f"Null MRP: {products['MRP'].isnull().sum()}")

# Fill nulls with category median
products['MRP'] = products.groupby('Category')['MRP'].transform(
    lambda x: x.fillna(x.median())
)

count    7.410000e+02
mean     1.819569e+06
std      2.628336e+06
min      3.489000e+03
25%      2.083750e+05
50%      5.717120e+05
75%      2.134462e+06
max      1.092242e+07
Name: MRP, dtype: float64
Null MRP: 59


In [46]:
#Clean Cost_Price

products['Cost_Price'] = products['Cost_Price'].apply(clean_price)

# Fix negative cost prices — take absolute value
negative_cost = (products['Cost_Price'] < 0).sum()
print(f"Negative cost prices found: {negative_cost}")
products['Cost_Price'] = products['Cost_Price'].abs()

# Cost cannot exceed MRP
invalid_cost = (products['Cost_Price'] > products['MRP']).sum()
print(f"Cost > MRP (before fix): {invalid_cost}")

# Fill nulls with category median
products['Cost_Price'] = products.groupby('Category')['Cost_Price'].transform(
    lambda x: x.fillna(x.median())
)

print(products[['MRP','Cost_Price']].describe())

Negative cost prices found: 0
Cost > MRP (before fix): 108
                MRP    Cost_Price
count  8.000000e+02  8.000000e+02
mean   1.794991e+06  1.232232e+06
std    2.576275e+06  1.841116e+06
min    3.489000e+03  1.664000e+03
25%    2.132282e+05  1.291095e+05
50%    5.315880e+05  3.245355e+05
75%    2.237204e+06  1.592038e+06
max    1.092242e+07  6.971979e+06


In [47]:
# Clean Discount_Pct and Selling_Price

# Clean Discount_Pct — remove % symbol
def clean_discount(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().replace('%','')
    try:
        d = float(val)
        # Valid range 0-70
        return d if 0 <= d <= 70 else np.nan
    except:
        return np.nan

products['Discount_Pct'] = products['Discount_Pct'].apply(clean_discount)

# Fill nulls with category median
products['Discount_Pct'] = products.groupby('Category')['Discount_Pct'].transform(
    lambda x: x.fillna(x.median())
)

# Recalculate Selling_Price from clean MRP and Discount
products['Selling_Price'] = products.apply(
    lambda row: round(row['MRP'] * (1 - row['Discount_Pct']/100), 2)
    if pd.notnull(row['MRP']) and pd.notnull(row['Discount_Pct']) else np.nan,
    axis=1
)

print(products[['MRP','Discount_Pct','Selling_Price']].head(10))

         MRP  Discount_Pct  Selling_Price
0  7207791.0           0.0     7207791.00
1   125704.0          10.0      113133.60
2   122841.0          10.0      110556.90
3  8155083.0          15.0     6931820.55
4   219372.0          30.0      153560.40
5  1073653.0          25.0      805239.75
6   117505.0           0.0      117505.00
7  8160479.0          25.0     6120359.25
8  6047019.0          20.0     4837615.20
9   141102.0           0.0      141102.00


In [48]:
# Clean Rating

products['Rating'] = pd.to_numeric(products['Rating'], errors='coerce')

# Valid range 1.0 to 5.0
print(f"Rating > 5: {(products['Rating'] > 5).sum()}")
print(f"Rating < 1: {(products['Rating'] < 1).sum()}")

products['Rating'] = products['Rating'].where(
    (products['Rating'] >= 1.0) & (products['Rating'] <= 5.0),
    np.nan
)

# Fill nulls with category median
products['Rating'] = products.groupby('Category')['Rating'].transform(
    lambda x: x.fillna(round(x.median(),1))
)

print(products['Rating'].describe())

Rating > 5: 97
Rating < 1: 53
count    800.000000
mean       3.817875
std        0.584911
min        1.700000
25%        3.600000
50%        3.900000
75%        4.125000
max        5.000000
Name: Rating, dtype: float64


In [49]:
#Clean Stock_Units

products['Stock_Units'] = pd.to_numeric(
    products['Stock_Units'], errors='coerce'
)

# Negative stock — data error — replace with 0
print(f"Negative stock: {(products['Stock_Units'] < 0).sum()}")
products['Stock_Units'] = products['Stock_Units'].clip(lower=0)

# Fill nulls with 0
products['Stock_Units'] = products['Stock_Units'].fillna(0).astype(int)

print(products['Stock_Units'].describe())

Negative stock: 69
count    800.00000
mean     195.74000
std      164.10538
min        0.00000
25%       13.00000
50%      187.50000
75%      341.00000
max      499.00000
Name: Stock_Units, dtype: float64


In [50]:
#Clean Weight_KG

def clean_weight(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower().replace('kg','').strip()
    try:
        w = float(val)
        return w if w > 0 else np.nan
    except:
        return np.nan

products['Weight_KG'] = products['Weight_KG'].apply(clean_weight)

# Fill nulls with subcategory median
products['Weight_KG'] = products.groupby('Sub_Category')['Weight_KG'].transform(
    lambda x: x.fillna(x.median())
)

# If still null fill with category median
products['Weight_KG'] = products.groupby('Category')['Weight_KG'].transform(
    lambda x: x.fillna(x.median())
)

print(products['Weight_KG'].describe())

count    800.000000
mean       1.151375
std        2.674002
min        0.020000
25%        0.180000
50%        0.390000
75%        0.710000
max       16.750000
Name: Weight_KG, dtype: float64


In [51]:
#Parse Launch_Date

def parse_date(val):
    if pd.isnull(val) or str(val).strip() in ['','NA','N/A','nan','null','-']:
        return np.nan
    val = str(val).strip()
    formats = ['%Y-%m-%d','%d/%m/%Y','%d-%m-%Y',
               '%m/%d/%Y','%d.%m.%Y','%Y/%m/%d']
    for fmt in formats:
        try:
            return datetime.strptime(val, fmt).strftime('%Y-%m-%d')
        except:
            continue
    return np.nan

products['Launch_Date'] = products['Launch_Date'].apply(parse_date)
products['Launch_Date'] = pd.to_datetime(products['Launch_Date'], errors='coerce')

# Fill missing with category median launch date
median_launch = products['Launch_Date'].dropna().sort_values()
mid_date = median_launch.iloc[len(median_launch)//2]
products['Launch_Date'] = products['Launch_Date'].fillna(mid_date)

print(products['Launch_Date'].describe())

count                              800
mean     2022-07-17 20:58:12.000000256
min                2020-01-01 00:00:00
25%                2021-07-24 18:00:00
50%                2022-07-15 00:00:00
75%                2023-08-13 12:00:00
max                2024-12-23 00:00:00
Name: Launch_Date, dtype: object


In [52]:
#Standardize Product_Status

def standardize_status(val):
    if pd.isnull(val):
        return 'Unknown'
    val = str(val).strip().lower()
    if 'active' in val and 'dis' not in val: return 'Active'
    if 'discontinue' in val:                 return 'Discontinued'
    if 'upcoming' in val:                    return 'Upcoming'
    if 'seasonal' in val:                    return 'Seasonal'
    if 'clearance' in val:                   return 'Clearance'
    if val in ['no','0','false']:            return 'Discontinued'
    return 'Active'   # default

products['Product_Status'] = products['Product_Status'].apply(
    standardize_status
)

print(products['Product_Status'].value_counts())

Product_Status
Active          352
Unknown         109
Clearance        88
Seasonal         86
Discontinued     83
Upcoming         82
Name: count, dtype: int64


In [53]:
# Outlier Treatment

def iqr_cap(series):
    Q1  = series.quantile(0.25)
    Q3  = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    print(f"  Bounds: {lower:.2f} to {upper:.2f}")
    outliers = ((series < lower) | (series > upper)).sum()
    print(f"  Outliers capped: {outliers}")
    return series.clip(lower=lower, upper=upper)

print("\nMRP outlier treatment:")
products['MRP'] = iqr_cap(products['MRP'])

print("\nCost_Price outlier treatment:")
products['Cost_Price'] = iqr_cap(products['Cost_Price'])

print("\nDiscount_Pct outlier treatment:")
products['Discount_Pct'] = iqr_cap(products['Discount_Pct'])


MRP outlier treatment:
  Bounds: -2822735.75 to 5273168.25
  Outliers capped: 102

Cost_Price outlier treatment:
  Bounds: -2065282.50 to 3786429.50
  Outliers capped: 101

Discount_Pct outlier treatment:
  Bounds: -17.50 to 42.50
  Outliers capped: 13


In [54]:
# Feature Engineering

today = pd.Timestamp.now()

# 1. Profit Per Unit
products['Profit_Per_Unit'] = round(
    products['Selling_Price'] - products['Cost_Price'], 2
)

# 2. Gross Margin Percent
products['Gross_Margin_Pct'] = round(
    (products['Profit_Per_Unit'] / products['Selling_Price']) * 100, 2
).clip(lower=-100, upper=100)

# 3. Is Loss Maker
products['Is_Loss_Maker'] = (
    products['Selling_Price'] < products['Cost_Price']
).astype(int)

print(f"Loss-making products: {products['Is_Loss_Maker'].sum()}")

# 4. Product Age in Days
products['Product_Age_Days'] = (
    today - products['Launch_Date']
).dt.days

# 5. Product Age Band
def age_band(days):
    if pd.isnull(days): return 'Unknown'
    if days < 180:   return 'New (< 6 months)'
    elif days < 365: return 'Growing (6-12 months)'
    elif days < 730: return 'Mature (1-2 years)'
    else:            return 'Established (2+ years)'

products['Product_Age_Band'] = products['Product_Age_Days'].apply(age_band)

# 6. Rating Band
def rating_band(r):
    if pd.isnull(r): return 'Unknown'
    if r >= 4.5:   return 'Excellent'
    elif r >= 4.0: return 'Good'
    elif r >= 3.0: return 'Average'
    else:          return 'Poor'

products['Rating_Band'] = products['Rating'].apply(rating_band)

# 7. Stock Status
def stock_status(s):
    if pd.isnull(s): return 'Unknown'
    if s == 0:     return 'Out of Stock'
    elif s <= 20:  return 'Low Stock'
    elif s <= 100: return 'Adequate'
    else:          return 'High Stock'

products['Stock_Status'] = products['Stock_Units'].apply(stock_status)

# 8. High Discount Flag
products['High_Discount_Flag'] = (
    products['Discount_Pct'] > 20
).astype(int)

# 9. Price Tier
def price_tier(mrp):
    if pd.isnull(mrp): return 'Unknown'
    if mrp < 500:      return 'Budget'
    elif mrp < 2000:   return 'Mid-Range'
    elif mrp < 10000:  return 'Premium'
    else:              return 'Luxury'

products['Price_Tier'] = products['MRP'].apply(price_tier)

# 10. Discount Impact Score
# How much revenue lost per unit due to discount
products['Discount_Amount'] = round(
    products['MRP'] - products['Selling_Price'], 2
)

print(products[['Product_ID','Profit_Per_Unit','Gross_Margin_Pct',
                'Is_Loss_Maker','Rating_Band','Stock_Status',
                'High_Discount_Flag','Price_Tier']].head(10))

Loss-making products: 152
  Product_ID  Profit_Per_Unit  Gross_Margin_Pct  Is_Loss_Maker Rating_Band  \
0  ELEC-1000       4183356.50             58.04              0     Average   
1  HOME-3000         40742.60             36.01              0     Average   
2  BEAU-4000        -52134.10            -47.16              1     Average   
3  ELEC-1001       3145391.05             45.38              0        Poor   
4  FASH-2000        -79551.60            -51.80              1     Average   
5  ELEC-1002         33687.75              4.18              0   Excellent   
6  HOME-3001         49573.00             42.19              0     Average   
7  ELEC-1003       2333929.75             38.13              0     Average   
8  ELEC-1004       1813180.70             37.48              0        Good   
9  HOME-3002         61496.00             43.58              0   Excellent   

   Stock_Status  High_Discount_Flag Price_Tier  
0    High Stock                   0     Luxury  
1    High Stock  

In [55]:
#Normalization

from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Min-Max — Rating and Discount_Pct
scaler_mm = MinMaxScaler()
products['Rating_Normalized']   = scaler_mm.fit_transform(products[['Rating']])
products['Discount_Normalized'] = scaler_mm.fit_transform(products[['Discount_Pct']])

# Z-Score — MRP and Profit_Per_Unit
scaler_z = StandardScaler()
products['MRP_Zscore']    = scaler_z.fit_transform(products[['MRP']])
products['Profit_Zscore'] = scaler_z.fit_transform(products[['Profit_Per_Unit']])

# Log Transform — MRP (right skewed)
products['MRP_Log']   = np.log1p(products['MRP'])
products['Cost_Log']  = np.log1p(products['Cost_Price'])
products['Stock_Log'] = np.log1p(products['Stock_Units'])

print(products[['Rating_Normalized','Discount_Normalized',
                'MRP_Zscore','MRP_Log']].describe())

       Rating_Normalized  Discount_Normalized    MRP_Zscore     MRP_Log
count         800.000000           800.000000  8.000000e+02  800.000000
mean            0.641780             0.332574 -6.439294e-17   13.270183
std             0.177246             0.239283  1.000626e+00    1.536468
min             0.000000             0.000000 -7.992999e-01    8.157657
25%             0.575758             0.117647 -6.869006e-01   12.270123
50%             0.666667             0.352941 -5.162914e-01   13.183626
75%             0.734848             0.470588  3.977490e-01   14.620726
max             1.000000             1.000000  2.024723e+00   15.478142


In [56]:
#Encoding Categorical Variables

# 1. One Hot Encoding — Category
cat_dummies = pd.get_dummies(
    products['Category'], prefix='Cat', drop_first=False
)
products = pd.concat([products, cat_dummies], axis=1)

# 2. One Hot Encoding — Price_Tier (ordinal encoding better)
tier_order = {'Budget':1,'Mid-Range':2,'Premium':3,'Luxury':4,'Unknown':0}
products['Price_Tier_Encoded'] = products['Price_Tier'].map(tier_order)

# 3. Ordinal Encoding — Rating_Band
rating_order = {'Poor':1,'Average':2,'Good':3,'Excellent':4,'Unknown':0}
products['Rating_Band_Encoded'] = products['Rating_Band'].map(rating_order)

# 4. Binary Encoding — Is_Active
products['Is_Active'] = (
    products['Product_Status'] == 'Active'
).astype(int)

# 5. Stock Status Encoding
stock_order = {'Out of Stock':0,'Low Stock':1,'Adequate':2,'High Stock':3,'Unknown':-1}
products['Stock_Status_Encoded'] = products['Stock_Status'].map(stock_order)

print(products[['Price_Tier_Encoded','Rating_Band_Encoded',
                'Is_Active','Stock_Status_Encoded']].head(10))

   Price_Tier_Encoded  Rating_Band_Encoded  Is_Active  Stock_Status_Encoded
0                   4                    2          0                     3
1                   4                    2          1                     3
2                   4                    2          1                     0
3                   4                    1          1                     3
4                   4                    2          1                     0
5                   4                    4          0                     3
6                   4                    2          0                     3
7                   4                    2          1                     2
8                   4                    3          1                     3
9                   4                    4          1                     0


In [57]:
# Final Validation
print("\n" + "="*55)
print("  PRODUCTS — FINAL DATA QUALITY REPORT")
print("="*55)
print(f"  Total rows          : {len(products):,}")
print(f"  Total columns       : {len(products.columns)}")
print(f"  Remaining nulls     : {products.isnull().sum().sum():,}")
print(f"  Duplicates          : {products.duplicated().sum()}")
print(f"  Loss-making products: {products['Is_Loss_Maker'].sum()}")
print(f"  Out of stock        : {(products['Stock_Status']=='Out of Stock').sum()}")
print(f"  High discount prods : {products['High_Discount_Flag'].sum()}")

print(f"\n  CATEGORY DISTRIBUTION:")
print(products['Category'].value_counts())

print(f"\n  PRICE TIER DISTRIBUTION:")
print(products['Price_Tier'].value_counts())

print(f"\n  RATING BAND DISTRIBUTION:")
print(products['Rating_Band'].value_counts())

print(f"\n  GROSS MARGIN BY CATEGORY:")
print(products.groupby('Category')['Gross_Margin_Pct'].describe().round(2))


  PRODUCTS — FINAL DATA QUALITY REPORT
  Total rows          : 800
  Total columns       : 39
  Remaining nulls     : 0
  Duplicates          : 0
  Loss-making products: 152
  Out of stock        : 190
  High discount prods : 149

  CATEGORY DISTRIBUTION:
Category
Electronics       275
Fashion           259
Home & Kitchen    152
Beauty            114
Name: count, dtype: int64

  PRICE TIER DISTRIBUTION:
Price_Tier
Luxury     792
Premium      8
Name: count, dtype: int64

  RATING BAND DISTRIBUTION:
Rating_Band
Average      484
Good         142
Excellent    108
Poor          66
Name: count, dtype: int64

  GROSS MARGIN BY CATEGORY:
                count   mean    std    min    25%    50%    75%    max
Category                                                              
Beauty          114.0  24.45  52.82 -100.0  28.30  40.03  50.39  95.70
Electronics     275.0  13.92  49.22 -100.0   2.52  20.57  41.84  99.40
Fashion         259.0  30.44  49.59 -100.0  24.23  43.55  56.23  96.28
Home &

In [58]:
# Save full cleaned version
products.to_csv('KartZone_Products_Clean.csv', index=False)
print("Saved → KartZone_Products_Clean.csv ✓")

# Save essential columns for SQL load
essential_cols = [
    'Product_ID','Product_Name','Category','Sub_Category','Brand',
    'MRP','Cost_Price','Discount_Pct','Selling_Price',
    'Launch_Date','Rating','Stock_Units','Weight_KG','Product_Status',
    'Profit_Per_Unit','Gross_Margin_Pct','Is_Loss_Maker',
    'Product_Age_Days','Product_Age_Band','Rating_Band',
    'Stock_Status','High_Discount_Flag','Price_Tier','Discount_Amount',
    'Is_Active'
]
products[essential_cols].to_csv('KartZone_Products_Final.csv', index=False)
print("Saved → KartZone_Products_Final.csv ✓")

Saved → KartZone_Products_Clean.csv ✓
Saved → KartZone_Products_Final.csv ✓


In [61]:
#orders
import pandas as pd
import numpy as np
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load
orders = pd.read_csv('Data Generation/KartZone_Orders.csv')

# First look
print(orders.shape)
print(orders.head())
print(orders.info())
print(orders.describe())
print(orders.columns.tolist())

(10170, 21)
        Order_ID Customer_ID Product_ID  Order_Date Expected_Delivery_Date  \
0  ORD-KZ-100001       C1181  ELEC-1230  23-10-2024                    NaN   
1  ORD-KZ-100002       C1939  ELEC-1765  16-10-2023             22.10.2023   
2  ORD-KZ-100003       C1772  FASH-2288  01-05-2023             2023-05-08   
3  ORD-KZ-100004       C1287  BEAU-4066  07-11-2024             11/10/2024   
4  ORD-KZ-100005       C1249  ELEC-1499  08/17/2024             08/22/2024   

        Category       City      Payment_Mode  Quantity       MRP  ...  \
0    Electronics    mumbai   Cash on Delivery       2.0  64321.13  ...   
1    Electronics     Bombay       NET BANKING       3.0  74959.82  ...   
2        FASHION  BANGALORE               Upi       2.0   1276.70  ...   
3  Personal Care  NEW DELHI       No Cost EMI       1.0    373.52  ...   
4    ELECTRONICS  hyderabad       Credit Card       4.0  95649.93  ...   

  Selling_Price  Final_Amount Unit_Cost  Total_Cost    Profit  \
0      51

In [62]:
#Understand Data Quality

# Missing values
print(orders.isnull().sum())
print(orders.isnull().sum() / len(orders) * 100)

# Duplicates
print(f"Duplicate rows: {orders.duplicated().sum()}")

# Blank rows
print(f"Blank rows: {orders.isnull().all(axis=1).sum()}")

# Unique values per column
print(orders.nunique())

# Sample messy values
for col in orders.columns:
    print(f"\n{col}:")
    print(orders[col].value_counts(dropna=False).head(8))

Order_ID                    20
Customer_ID                 20
Product_ID                  20
Order_Date                  20
Expected_Delivery_Date    1840
Category                    20
City                        20
Payment_Mode                20
Quantity                   851
MRP                         20
Discount_Pct               843
Selling_Price               20
Final_Amount                20
Unit_Cost                   20
Total_Cost                  20
Profit                    1270
Profit_Margin_Pct           20
Order_Status                20
Return_Reason             9038
Coupon_Code               2833
Is_First_Order             227
dtype: int64
Order_ID                   0.196657
Customer_ID                0.196657
Product_ID                 0.196657
Order_Date                 0.196657
Expected_Delivery_Date    18.092429
Category                   0.196657
City                       0.196657
Payment_Mode               0.196657
Quantity                   8.367748
MRP         

In [63]:
#Remove Duplicates and Blank Rows

print(f"Before: {len(orders)}")

# Remove blank rows
orders = orders.dropna(how='all')

# Remove exact duplicates
orders = orders.drop_duplicates()

# Remove duplicate Order_IDs — keep first
orders = orders.drop_duplicates(subset=['Order_ID'], keep='first')

# Reset index
orders = orders.reset_index(drop=True)

print(f"After: {len(orders)}")


Before: 10170
After: 10000


In [64]:
 #Clean Order_ID

# Strip whitespace
orders['Order_ID'] = orders['Order_ID'].astype(str).str.strip()

# Check for nulls or invalid
print(f"Null Order_IDs: {orders['Order_ID'].isnull().sum()}")
print(f"Sample IDs: {orders['Order_ID'].head(5).tolist()}")

# Remove rows with null Order_ID
orders = orders.dropna(subset=['Order_ID'])
orders = orders.reset_index(drop=True)

Null Order_IDs: 0
Sample IDs: ['ORD-KZ-100001', 'ORD-KZ-100002', 'ORD-KZ-100003', 'ORD-KZ-100004', 'ORD-KZ-100005']


In [65]:
#Validate Customer_ID and Product_ID
# Load valid IDs from clean tables
try:
    valid_customers = pd.read_csv('KartZone_Customers_Final.csv')['Customer_ID'].tolist()
    valid_products  = pd.read_csv('KartZone_Products_Final.csv')['Product_ID'].tolist()
except:
    valid_customers = [f'C{i}' for i in range(1001,2001)]
    valid_products  = None

# Flag orphan Customer_IDs
orders['Is_Valid_Customer'] = orders['Customer_ID'].isin(valid_customers)
print(f"Orphan customers: {(~orders['Is_Valid_Customer']).sum()}")

# Flag orphan Product_IDs
if valid_products:
    orders['Is_Valid_Product'] = orders['Product_ID'].isin(valid_products)
    print(f"Orphan products: {(~orders['Is_Valid_Product']).sum()}")

# Remove orphan rows
orders = orders[orders['Is_Valid_Customer']].reset_index(drop=True)
orders = orders.drop(columns=['Is_Valid_Customer'], errors='ignore')

print(f"Orders after orphan removal: {len(orders)}")


Orphan customers: 209
Orphan products: 7259
Orders after orphan removal: 9791


In [66]:
#Parse Date Columns

def parse_date(val):
    if pd.isnull(val) or str(val).strip() in ['','NA','N/A','nan','null','-']:
        return np.nan
    val = str(val).strip()
    formats = ['%Y-%m-%d','%d/%m/%Y','%d-%m-%Y',
               '%m/%d/%Y','%d.%m.%Y','%Y/%m/%d']
    for fmt in formats:
        try:
            return datetime.strptime(val, fmt).strftime('%Y-%m-%d')
        except:
            continue
    return np.nan

date_cols = ['Order_Date','Expected_Delivery_Date']
for col in date_cols:
    orders[col] = orders[col].apply(parse_date)
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Order_Date is critical — drop rows where it is null
print(f"Null Order_Date: {orders['Order_Date'].isnull().sum()}")
orders = orders.dropna(subset=['Order_Date']).reset_index(drop=True)

print(orders[date_cols].dtypes)
print(orders[date_cols].isnull().sum())

Null Order_Date: 0
Order_Date                datetime64[ns]
Expected_Delivery_Date    datetime64[ns]
dtype: object
Order_Date                   0
Expected_Delivery_Date    1756
dtype: int64


In [67]:
# Standardize Order_Status

def standardize_status(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    if val in ['delivered','deliverd','complete','completed']:
        return 'Delivered'
    if val in ['returned','return','refunded']:
        return 'Returned'
    if val in ['cancelled','canceled','cancel']:
        return 'Cancelled'
    if val in ['pending','in transit','processing','shipped']:
        return 'Pending'
    return np.nan

orders['Order_Status'] = orders['Order_Status'].apply(standardize_status)

# Fill nulls with mode
orders['Order_Status'] = orders['Order_Status'].fillna(
    orders['Order_Status'].mode()[0]
)

print(orders['Order_Status'].value_counts())

Order_Status
Delivered    6424
Returned     1416
Cancelled    1201
Pending       750
Name: count, dtype: int64


In [68]:
#Standardize Payment_Mode
def standardize_payment(val):
    if pd.isnull(val):
        return 'Unknown'
    val = str(val).strip().lower()
    if 'upi' in val:                          return 'UPI'
    if 'credit' in val or val == 'cc':        return 'Credit Card'
    if 'debit' in val or val == 'dc':         return 'Debit Card'
    if 'cod' in val or 'cash' in val:         return 'COD'
    if 'net' in val or val == 'nb':           return 'Net Banking'
    if 'emi' in val:                          return 'EMI'
    if 'wallet' in val or 'paytm' in val or 'phonepe' in val: return 'Wallet'
    return 'Other'

orders['Payment_Mode'] = orders['Payment_Mode'].apply(standardize_payment)

print(orders['Payment_Mode'].value_counts())

Payment_Mode
UPI            3456
COD            1734
Credit Card    1459
Debit Card     1207
Net Banking     767
EMI             691
Wallet          384
Other            93
Name: count, dtype: int64


In [69]:
#Standardize Category

def standardize_category(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    if 'electron' in val:                    return 'Electronics'
    if 'fashion' in val or 'cloth' in val:   return 'Fashion'
    if 'home' in val or 'kitchen' in val:    return 'Home & Kitchen'
    if 'beauty' in val or 'personal' in val: return 'Beauty'
    return np.nan

orders['Category'] = orders['Category'].apply(standardize_category)

# Derive from Product_ID prefix if still null
def derive_cat(pid):
    if pd.isnull(pid): return np.nan
    pid = str(pid).upper()
    if pid.startswith('ELEC'): return 'Electronics'
    if pid.startswith('FASH'): return 'Fashion'
    if pid.startswith('HOME'): return 'Home & Kitchen'
    if pid.startswith('BEAU'): return 'Beauty'
    return np.nan

orders['Category'] = orders.apply(
    lambda row: derive_cat(row['Product_ID'])
    if pd.isnull(row['Category']) else row['Category'],
    axis=1
)

print(orders['Category'].value_counts())


Category
Electronics       3459
Fashion           2894
Home & Kitchen    1929
Beauty            1485
Name: count, dtype: int64


In [70]:
#Standardize City
def standardize_city(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    city_map = {
        'mumbai':'Mumbai','bombay':'Mumbai',
        'delhi':'Delhi','new delhi':'Delhi',
        'bangalore':'Bangalore','bengaluru':'Bangalore','banglore':'Bangalore',
        'chennai':'Chennai','madras':'Chennai',
        'hyderabad':'Hyderabad','hyd':'Hyderabad','hydrabad':'Hyderabad',
        'pune':'Pune','poona':'Pune'
    }
    for key, standard in city_map.items():
        if key in val:
            return standard
    return np.nan

orders['City'] = orders['City'].apply(standardize_city)
orders['City'] = orders['City'].fillna(orders['City'].mode()[0])

print(orders['City'].value_counts())

City
Mumbai       2452
Delhi        2143
Bangalore    1971
Chennai      1236
Hyderabad    1175
Pune          814
Name: count, dtype: int64


In [71]:
#Clean Quantity
orders['Quantity'] = pd.to_numeric(orders['Quantity'], errors='coerce')

# Valid range 1-10
print(f"Invalid quantity (<1): {(orders['Quantity'] < 1).sum()}")
print(f"Invalid quantity (>10): {(orders['Quantity'] > 10).sum()}")

orders['Quantity'] = orders['Quantity'].where(
    (orders['Quantity'] >= 1) & (orders['Quantity'] <= 10),
    np.nan
)

# Fill nulls with 1 (most common)
orders['Quantity'] = orders['Quantity'].fillna(1).astype(int)

print(orders['Quantity'].value_counts())


Invalid quantity (<1): 0
Invalid quantity (>10): 0
Quantity
1    5299
2    2240
3    1093
4     707
5     452
Name: count, dtype: int64


In [72]:
#Clean MRP and Pricing Columns

def clean_price(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip()
    val = re.sub(r'[₹Rs\.INR,/\-\s]', '', val)
    try:
        result = float(val)
        return result if result > 0 else np.nan
    except:
        return np.nan

# Clean all price columns
price_cols = ['MRP','Selling_Price','Final_Amount','Unit_Cost','Total_Cost']
for col in price_cols:
    if col in orders.columns:
        orders[col] = orders[col].apply(clean_price)

print(orders[price_cols].describe())

                MRP  Selling_Price  Final_Amount     Unit_Cost    Total_Cost
count  9.791000e+03   9.791000e+03  9.127000e+03  9.791000e+03  9.791000e+03
mean   1.907959e+06   1.596325e+06  2.785134e+06  1.288564e+06  2.242387e+06
std    2.697130e+06   2.280830e+06  5.033410e+06  1.925989e+06  4.209376e+06
min    1.348000e+03   1.218000e+03  1.218000e+03  8.820000e+02  8.820000e+02
25%    2.313355e+05   1.954835e+05  2.470725e+05  1.213950e+05  1.550970e+05
50%    6.182950e+05   5.149120e+05  7.393120e+05  3.240750e+05  4.598390e+05
75%    2.322688e+06   1.816188e+06  2.716716e+06  1.562540e+06  2.054048e+06
max    1.110996e+07   1.105659e+07  5.528294e+07  6.997675e+06  3.368998e+07


In [73]:
#Recalculate Final_Amount

# Final_Amount = Selling_Price * Quantity
# Recalculate from clean values to fix negatives and inconsistencies

# First fix negative Final_Amount
neg_final = (orders['Final_Amount'] < 0).sum()
print(f"Negative Final_Amount: {neg_final}")

# Recalculate where Final_Amount is null or negative
mask = orders['Final_Amount'].isnull() | (orders['Final_Amount'] < 0)
orders.loc[mask, 'Final_Amount'] = (
    orders.loc[mask, 'Selling_Price'] * orders.loc[mask, 'Quantity']
).round(2)

# Recalculate Total_Cost
orders['Total_Cost'] = (
    orders['Unit_Cost'] * orders['Quantity']
).round(2)

print(f"Remaining null Final_Amount: {orders['Final_Amount'].isnull().sum()}")

Negative Final_Amount: 0
Remaining null Final_Amount: 0


In [74]:
# Clean Discount_Pct

def clean_discount(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().replace('%','')
    try:
        d = float(val)
        return d if 0 <= d <= 70 else np.nan
    except:
        return np.nan

orders['Discount_Pct'] = orders['Discount_Pct'].apply(clean_discount)

# Fill nulls with category median
orders['Discount_Pct'] = orders.groupby('Category')['Discount_Pct'].transform(
    lambda x: x.fillna(x.median())
)

print(orders['Discount_Pct'].describe())

count    9767.000000
mean       15.031740
std        12.056085
min         0.000000
25%         5.000000
50%        15.000000
75%        20.000000
max        70.000000
Name: Discount_Pct, dtype: float64


In [75]:
# Recalculate and Clean Profit
# Profit = Final_Amount - Total_Cost
# Recalculate from clean values

orders['Profit'] = (
    orders['Final_Amount'] - orders['Total_Cost']
).round(2)

# Profit_Margin_Pct
orders['Profit_Margin_Pct'] = (
    orders['Profit'] / orders['Final_Amount'] * 100
).round(2)

# Cap extreme margins
orders['Profit_Margin_Pct'] = orders['Profit_Margin_Pct'].clip(
    lower=-100, upper=100
)

print(orders[['Final_Amount','Total_Cost','Profit','Profit_Margin_Pct']].describe())


       Final_Amount    Total_Cost        Profit  Profit_Margin_Pct
count  9.791000e+03  9.791000e+03  9.791000e+03        9791.000000
mean   2.791943e+06  2.368568e+06  4.233747e+05          18.963157
std    5.039448e+06  4.372583e+06  3.071167e+06          53.107064
min    1.218000e+03  8.820000e+02 -3.022598e+07        -100.000000
25%    2.502900e+05  1.717635e+05  3.622300e+04          10.395000
50%    7.438830e+05  4.843680e+05  2.078250e+05          30.090000
75%    2.769124e+06  2.321954e+06  6.483805e+05          48.345000
max    5.528294e+07  3.464113e+07  3.806420e+07          99.140000


In [76]:
# Clean Return_Reason

# Standardize missing encodings
orders['Return_Reason'] = orders['Return_Reason'].replace(
    ['nan','NaN','NA','N/A','None','','null','-'], np.nan
)

# Return reason should only exist for Returned orders
non_returned_with_reason = (
    (orders['Order_Status'] != 'Returned') &
    orders['Return_Reason'].notna()
).sum()
print(f"Non-returned orders with return reason: {non_returned_with_reason}")

# Clear return reasons for non-returned orders
orders.loc[orders['Order_Status'] != 'Returned', 'Return_Reason'] = np.nan

# Fill missing return reasons for returned orders
orders.loc[
    (orders['Order_Status'] == 'Returned') &
    (orders['Return_Reason'].isnull()),
    'Return_Reason'
] = 'Reason Not Provided'

print(orders['Return_Reason'].value_counts())

Non-returned orders with return reason: 0
Return_Reason
Reason Not Provided        328
Delayed Delivery           131
Not as Described           126
Duplicate Order            117
Changed Mind               113
Wrong Item Delivered       110
Product Defective          104
Product Damaged             99
Better Price Available      97
Size/Fit Issue              96
Quality Not as Expected     95
Name: count, dtype: int64


In [77]:
#Clean Coupon_Code

# Standardize missing
orders['Coupon_Code'] = orders['Coupon_Code'].replace(
    ['nan','NaN','NA','N/A','None','','null','-'], np.nan
)

# Uppercase all valid coupons
orders['Coupon_Code'] = orders['Coupon_Code'].str.upper()

print(orders['Coupon_Code'].value_counts().head(10))
print(f"Orders with coupon: {orders['Coupon_Code'].notna().sum()}")

Coupon_Code
KARTZONE15    729
SAVE10        727
APP25         721
FEST20        720
WELCOME15     711
FLASH20       707
NEWUSER10     706
NEWYEAR10     690
DIWALI25      680
SALE30        676
Name: count, dtype: int64
Orders with coupon: 7067


In [78]:
#Clean Is_First_Order

def clean_bool(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    if val in ['yes','1','true','y']:  return 1
    if val in ['no','0','false','n']:  return 0
    return np.nan

orders['Is_First_Order'] = orders['Is_First_Order'].apply(clean_bool)
orders['Is_First_Order'] = orders['Is_First_Order'].fillna(0).astype(int)

print(orders['Is_First_Order'].value_counts())

Is_First_Order
0    7298
1    2493
Name: count, dtype: int64


In [79]:
#Outlier Treatment

def iqr_cap(series, col_name):
    Q1  = series.quantile(0.25)
    Q3  = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((series < lower) | (series > upper)).sum()
    print(f"  {col_name}: {outliers} outliers | bounds [{lower:.2f}, {upper:.2f}]")
    return series.clip(lower=lower, upper=upper)

print("\nOutlier treatment:")
orders['Final_Amount']     = iqr_cap(orders['Final_Amount'], 'Final_Amount')
orders['Discount_Pct']     = iqr_cap(orders['Discount_Pct'], 'Discount_Pct')
orders['Profit']           = iqr_cap(orders['Profit'], 'Profit')
orders['Profit_Margin_Pct']= iqr_cap(orders['Profit_Margin_Pct'], 'Profit_Margin_Pct')


Outlier treatment:
  Final_Amount: 1270 outliers | bounds [-3527961.00, 6547375.00]
  Discount_Pct: 363 outliers | bounds [-17.50, 42.50]
  Profit: 1775 outliers | bounds [-882013.25, 1566616.75]
  Profit_Margin_Pct: 1330 outliers | bounds [-46.53, 105.27]


In [80]:
#Feature Engineering

today = pd.Timestamp.now()

# 1. Date-based features
orders['Order_Year']    = orders['Order_Date'].dt.year
orders['Order_Month']   = orders['Order_Date'].dt.month
orders['Order_Quarter'] = orders['Order_Date'].dt.quarter
orders['Order_DayOfWeek'] = orders['Order_Date'].dt.day_name()
orders['Is_Weekend']    = orders['Order_Date'].dt.dayofweek.isin([5,6]).astype(int)

# 2. Festive Season Flag — Oct Nov Dec
orders['Is_Festive_Season'] = orders['Order_Month'].isin([10,11,12]).astype(int)

# 3. Order Status Flags
orders['Is_Delivered']  = (orders['Order_Status'] == 'Delivered').astype(int)
orders['Is_Returned']   = (orders['Order_Status'] == 'Returned').astype(int)
orders['Is_Cancelled']  = (orders['Order_Status'] == 'Cancelled').astype(int)

# 4. High Discount Flag
orders['High_Discount_Flag'] = (orders['Discount_Pct'] > 20).astype(int)

# 5. Coupon Used Flag
orders['Coupon_Used'] = orders['Coupon_Code'].notna().astype(int)

# 6. Is Loss Order
orders['Is_Loss_Order'] = (orders['Profit'] < 0).astype(int)
print(f"Loss-making orders: {orders['Is_Loss_Order'].sum()}")

# 7. Order Value Band
def order_value_band(amt):
    if pd.isnull(amt): return 'Unknown'
    if amt < 500:        return 'Low (< 500)'
    elif amt < 2000:     return 'Medium (500-2K)'
    elif amt < 10000:    return 'High (2K-10K)'
    else:                return 'Premium (10K+)'

orders['Order_Value_Band'] = orders['Final_Amount'].apply(order_value_band)

# 8. COD Flag
orders['Is_COD'] = (orders['Payment_Mode'] == 'COD').astype(int)

# 9. Days to Expected Delivery
orders['Expected_Delivery_Days'] = (
    orders['Expected_Delivery_Date'] - orders['Order_Date']
).dt.days

# 10. Revenue Contribution Percent
total_revenue = orders['Final_Amount'].sum()
orders['Revenue_Contribution_Pct'] = round(
    orders['Final_Amount'] / total_revenue * 100, 4
)

print(orders[['Is_Festive_Season','Is_Returned','High_Discount_Flag',
              'Is_Loss_Order','Order_Value_Band','Is_COD']].head(10))

Loss-making orders: 1791
   Is_Festive_Season  Is_Returned  High_Discount_Flag  Is_Loss_Order  \
0                  1            0                   0              1   
1                  1            0                   0              0   
2                  0            1                   1              0   
3                  1            0                   0              0   
4                  0            0                   0              0   
5                  1            0                   0              0   
6                  0            0                   0              0   
7                  0            0                   0              0   
8                  0            0                   1              0   
9                  0            0                   0              0   

  Order_Value_Band  Is_COD  
0   Premium (10K+)       1  
1   Premium (10K+)       0  
2   Premium (10K+)       0  
3   Premium (10K+)       0  
4   Premium (10K+)       0  
5   Prem

In [81]:
# Normalization
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Min-Max Normalization
scaler_mm = MinMaxScaler()
orders['Final_Amount_Normalized'] = scaler_mm.fit_transform(
    orders[['Final_Amount']]
)
orders['Discount_Normalized'] = scaler_mm.fit_transform(
    orders[['Discount_Pct']]
)

# Z-Score Standardization
scaler_z = StandardScaler()
orders['Profit_Zscore'] = scaler_z.fit_transform(
    orders[['Profit']]
)

# Log Transformation — Final_Amount is right skewed
orders['Final_Amount_Log'] = np.log1p(orders['Final_Amount'])
orders['Profit_Log']       = np.log1p(orders['Profit'].clip(lower=0))

print(orders[['Final_Amount_Normalized','Discount_Normalized',
              'Profit_Zscore','Final_Amount_Log']].describe())

       Final_Amount_Normalized  Discount_Normalized  Profit_Zscore  \
count              9791.000000          9767.000000   9.791000e+03   
mean                  0.288623             0.345031   4.717115e-17   
std                   0.348677             0.257513   1.000051e+00   
min                   0.000000             0.000000  -1.909301e+00   
25%                   0.038049             0.117647  -4.888814e-01   
50%                   0.113451             0.352941  -2.234303e-01   
75%                   0.422829             0.470588   4.580649e-01   
max                   1.000000             1.000000   1.878484e+00   

       Final_Amount_Log  
count       9791.000000  
mean          13.497343  
std            1.591268  
min            7.105786  
25%           12.430380  
50%           13.519640  
75%           14.834042  
max           15.694575  


In [82]:
#Encoding Categorical Variables

# 1. One Hot Encoding — Category
cat_dummies = pd.get_dummies(
    orders['Category'], prefix='Cat', drop_first=False
)
orders = pd.concat([orders, cat_dummies], axis=1)

# 2. One Hot Encoding — Payment_Mode
pay_dummies = pd.get_dummies(
    orders['Payment_Mode'], prefix='Pay', drop_first=True
)
orders = pd.concat([orders, pay_dummies], axis=1)

# 3. Label Encoding — Order_Status
status_order = {'Cancelled':0,'Pending':1,'Returned':2,'Delivered':3}
orders['Status_Encoded'] = orders['Order_Status'].map(status_order)

# 4. Month Encoding — Cyclical (important for seasonality)
orders['Month_Sin'] = np.sin(2 * np.pi * orders['Order_Month'] / 12)
orders['Month_Cos'] = np.cos(2 * np.pi * orders['Order_Month'] / 12)

# 5. Day of Week Encoding — Cyclical
orders['DayOfWeek_Num'] = orders['Order_Date'].dt.dayofweek
orders['Day_Sin'] = np.sin(2 * np.pi * orders['DayOfWeek_Num'] / 7)
orders['Day_Cos'] = np.cos(2 * np.pi * orders['DayOfWeek_Num'] / 7)

print(f"Shape after encoding: {orders.shape}")
print(orders[['Status_Encoded','Month_Sin','Month_Cos']].head())

Shape after encoding: (9791, 60)
   Status_Encoded  Month_Sin  Month_Cos
0               3  -0.866025   0.500000
1               0  -0.866025   0.500000
2               2   0.500000  -0.866025
3               3  -0.500000   0.866025
4               3  -0.866025  -0.500000


In [83]:
#Final Validation

print("\n" + "="*55)
print("  ORDERS — FINAL DATA QUALITY REPORT")
print("="*55)
print(f"  Total rows          : {len(orders):,}")
print(f"  Total columns       : {len(orders.columns)}")
print(f"  Remaining nulls     : {orders.isnull().sum().sum():,}")
print(f"  Duplicates          : {orders.duplicated().sum()}")

print(f"\n  BUSINESS METRICS:")
total_rev  = orders['Final_Amount'].sum()
total_prof = orders['Profit'].sum()
margin     = total_prof / total_rev * 100
print(f"  Total Revenue       : ₹{total_rev:,.2f}")
print(f"  Total Profit        : ₹{total_prof:,.2f}")
print(f"  Overall Margin      : {margin:.2f}%")
print(f"  Loss-making orders  : {orders['Is_Loss_Order'].sum():,}")
print(f"  Return rate         : {orders['Is_Returned'].mean()*100:.2f}%")
print(f"  Cancellation rate   : {orders['Is_Cancelled'].mean()*100:.2f}%")
print(f"  COD orders          : {orders['Is_COD'].mean()*100:.2f}%")
print(f"  Festive orders      : {orders['Is_Festive_Season'].mean()*100:.2f}%")

print(f"\n  ORDER STATUS:")
print(orders['Order_Status'].value_counts())

print(f"\n  CATEGORY REVENUE:")
print(orders.groupby('Category')['Final_Amount'].sum().sort_values(ascending=False))

print(f"\n  MONTHLY TREND:")
print(orders.groupby('Order_Month')['Final_Amount'].sum().round(0))


  ORDERS — FINAL DATA QUALITY REPORT
  Total rows          : 9,791
  Total columns       : 60
  Remaining nulls     : 14,683
  Duplicates          : 0

  BUSINESS METRICS:
  Total Revenue       : ₹18,510,785,762.00
  Total Profit        : ₹3,449,002,084.25
  Overall Margin      : 18.63%
  Loss-making orders  : 1,791
  Return rate         : 14.46%
  Cancellation rate   : 12.27%
  COD orders          : 17.71%
  Festive orders      : 44.12%

  ORDER STATUS:
Order_Status
Delivered    6424
Returned     1416
Cancelled    1201
Pending       750
Name: count, dtype: int64

  CATEGORY REVENUE:
Category
Electronics       1.443697e+10
Fashion           2.008834e+09
Home & Kitchen    1.632805e+09
Beauty            4.102286e+08
Name: Final_Amount, dtype: float64

  MONTHLY TREND:
Order_Month
1     1.187576e+09
2     1.003217e+09
3     1.193318e+09
4     1.144012e+09
5     1.134137e+09
6     8.658086e+08
7     1.101508e+09
8     1.203367e+09
9     1.408808e+09
10    2.705363e+09
11    3.261857e+09
1

In [84]:
# Save full cleaned version
orders.to_csv('KartZone_Orders_Clean.csv', index=False)
print("Saved → KartZone_Orders_Clean.csv ✓")

# Essential columns for SQL load
essential_cols = [
    'Order_ID','Customer_ID','Product_ID','Order_Date',
    'Expected_Delivery_Date','Category','City','Payment_Mode',
    'Quantity','MRP','Discount_Pct','Selling_Price',
    'Final_Amount','Unit_Cost','Total_Cost','Profit',
    'Profit_Margin_Pct','Order_Status','Return_Reason',
    'Coupon_Code','Is_First_Order',
    'Order_Year','Order_Month','Order_Quarter',
    'Is_Weekend','Is_Festive_Season',
    'Is_Delivered','Is_Returned','Is_Cancelled',
    'High_Discount_Flag','Coupon_Used','Is_Loss_Order',
    'Order_Value_Band','Is_COD','Revenue_Contribution_Pct'
]
orders[essential_cols].to_csv('KartZone_Orders_Final.csv', index=False)
print("Saved → KartZone_Orders_Final.csv ✓")

Saved → KartZone_Orders_Clean.csv ✓
Saved → KartZone_Orders_Final.csv ✓


In [86]:
#Deliveries Data

import pandas as pd
import numpy as np
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load
deliveries = pd.read_csv('Data Generation/KartZone_Deliveries.csv')

# First look
print(deliveries.shape)
print(deliveries.head())
print(deliveries.info())
print(deliveries.describe())
print(deliveries.columns.tolist())


(10115, 14)
  Delivery_ID       Order_ID   Delivery_Partner Warehouse_City Pickup_Date  \
0  DEL-000001  ORD-KZ-100001  Delhivery Pvt Ltd       Pune Hub         NaN   
1  DEL-000002  ORD-KZ-100002              EKART  Bangalore Hub  28-06-2024   
2  DEL-000003  ORD-KZ-100003          Blue Dart            NaN  12-01-2023   
3  DEL-000004  ORD-KZ-100004          DELHIVERY    Chennai Hub         NaN   
4  DEL-000005  ORD-KZ-100005       DTDC Courier  Bangalore Hub  2024-06-17   

   Expected_Days  Actual_Delivery_Days  Delay_Days Actual_Delivery_Date  \
0            5.0                   6.0         1.0                  NaN   
1            3.0                   2.0         NaN           2024-06-29   
2            3.0                   3.0         0.0           2023-01-14   
3            3.0                   NaN         NaN           11/03/2023   
4            5.0                   9.0         NaN                  NaN   

  Delivery_Status  Delivery_Attempts          Failure_Reason Deliver

In [87]:
# Understand Data Quality

# Missing values
print(deliveries.isnull().sum())
print(deliveries.isnull().sum() / len(deliveries) * 100)

# Duplicates
print(f"Duplicate rows: {deliveries.duplicated().sum()}")

# Blank rows
print(f"Blank rows: {deliveries.isnull().all(axis=1).sum()}")

# Unique values per column
print(deliveries.nunique())

# Sample messy values per column
for col in deliveries.columns:
    print(f"\n{col}:")
    print(deliveries[col].value_counts(dropna=False).head(8))

Delivery_ID               15
Order_ID                  15
Delivery_Partner         493
Warehouse_City          1031
Pickup_Date             1835
Expected_Days             15
Actual_Delivery_Days    1732
Delay_Days              1188
Actual_Delivery_Date    2046
Delivery_Status           15
Delivery_Attempts       1056
Failure_Reason          9727
Delivery_Cost           1851
Customer_Rating         2063
dtype: int64
Delivery_ID              0.148295
Order_ID                 0.148295
Delivery_Partner         4.873950
Warehouse_City          10.192783
Pickup_Date             18.141374
Expected_Days            0.148295
Actual_Delivery_Days    17.123085
Delay_Days              11.744933
Actual_Delivery_Date    20.227385
Delivery_Status          0.148295
Delivery_Attempts       10.439941
Failure_Reason          96.164113
Delivery_Cost           18.299555
Customer_Rating         20.395452
dtype: float64
Duplicate rows: 114
Blank rows: 15
Delivery_ID             10000
Order_ID                1

In [88]:
# Remove Duplicates and Blank Rows

print(f"Before: {len(deliveries)}")

# Remove blank rows
deliveries = deliveries.dropna(how='all')

# Remove exact duplicates
deliveries = deliveries.drop_duplicates()

# Remove duplicate Delivery_IDs
deliveries = deliveries.drop_duplicates(
    subset=['Delivery_ID'], keep='first'
)

# Remove duplicate Order_IDs — one delivery per order
deliveries = deliveries.drop_duplicates(
    subset=['Order_ID'], keep='first'
)

# Reset index
deliveries = deliveries.reset_index(drop=True)

print(f"After: {len(deliveries)}")

Before: 10115
After: 10000


In [89]:
#Clean Delivery_ID and Order_ID

# Strip whitespace
deliveries['Delivery_ID'] = deliveries['Delivery_ID'].astype(str).str.strip()
deliveries['Order_ID']    = deliveries['Order_ID'].astype(str).str.strip()

# Validate Order_ID — must match KartZone format
valid_format = deliveries['Order_ID'].str.match(r'^ORD-KZ-\d+$', na=False)
print(f"Valid Order_ID format: {valid_format.sum()}")
print(f"Invalid Order_ID format: {(~valid_format).sum()}")

# Remove rows with invalid Order_IDs
deliveries = deliveries[valid_format].reset_index(drop=True)

print(f"After ID validation: {len(deliveries)}")

Valid Order_ID format: 10000
Invalid Order_ID format: 0
After ID validation: 10000


In [90]:
#Validate Foreign Keys

# Check Order_IDs exist in Orders table
try:
    orders_clean = pd.read_csv('KartZone_Orders_Final.csv')
    valid_order_ids = orders_clean['Order_ID'].tolist()

    orphan_mask = ~deliveries['Order_ID'].isin(valid_order_ids)
    print(f"Orphan Order_IDs in Deliveries: {orphan_mask.sum()}")

    # Remove orphan deliveries
    deliveries = deliveries[~orphan_mask].reset_index(drop=True)
    print(f"After orphan removal: {len(deliveries)}")
except:
    print("Orders table not found — skipping foreign key validation")

Orphan Order_IDs in Deliveries: 209
After orphan removal: 9791


In [91]:
#Standardize Delivery_Partner

def standardize_partner(val):
    if pd.isnull(val):
        return 'Unknown'
    val = str(val).strip().lower()
    if 'delhivery' in val or val == 'dlv':
        return 'Delhivery'
    if 'blue dart' in val or 'bluedart' in val or 'blue-dart' in val:
        return 'Blue Dart'
    if 'dtdc' in val or 'd.t.d.c' in val:
        return 'DTDC'
    if 'ekart' in val:
        return 'Ekart'
    if 'xpressbees' in val or 'xpress bees' in val or val == 'xb':
        return 'XpressBees'
    if 'amazon' in val or 'amzn' in val:
        return 'Amazon Logistics'
    return 'Unknown'

deliveries['Delivery_Partner'] = deliveries['Delivery_Partner'].apply(
    standardize_partner
)

print(deliveries['Delivery_Partner'].value_counts())
print(f"Unknown partners: {(deliveries['Delivery_Partner']=='Unknown').sum()}")


Delivery_Partner
Delhivery           2554
Ekart               1855
Blue Dart           1723
DTDC                1448
XpressBees           927
Amazon Logistics     825
Unknown              459
Name: count, dtype: int64
Unknown partners: 459


In [92]:
#Standardize Warehouse_City
def standardize_city(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    city_map = {
        'mumbai': 'Mumbai Hub',
        'delhi':  'Delhi Hub',
        'bangalore': 'Bangalore Hub',
        'bengaluru': 'Bangalore Hub',
        'chennai': 'Chennai Hub',
        'hyderabad': 'Hyderabad Hub',
        'pune': 'Pune Hub'
    }
    for key, standard in city_map.items():
        if key in val:
            return standard
    return np.nan

deliveries['Warehouse_City'] = deliveries['Warehouse_City'].apply(
    standardize_city
)

# Fill missing with mode
deliveries['Warehouse_City'] = deliveries['Warehouse_City'].fillna(
    deliveries['Warehouse_City'].mode()[0]
)

print(deliveries['Warehouse_City'].value_counts())

Warehouse_City
Pune Hub         2485
Chennai Hub      1479
Hyderabad Hub    1479
Mumbai Hub       1472
Delhi Hub        1444
Bangalore Hub    1432
Name: count, dtype: int64


In [93]:
#Parse Date Columns

def parse_date(val):
    if pd.isnull(val) or str(val).strip() in ['','NA','N/A','nan','null','-']:
        return np.nan
    val = str(val).strip()
    formats = ['%Y-%m-%d','%d/%m/%Y','%d-%m-%Y',
               '%m/%d/%Y','%d.%m.%Y','%Y/%m/%d']
    for fmt in formats:
        try:
            return datetime.strptime(val, fmt).strftime('%Y-%m-%d')
        except:
            continue
    return np.nan

date_cols = ['Pickup_Date','Actual_Delivery_Date']
for col in date_cols:
    deliveries[col] = deliveries[col].apply(parse_date)
    deliveries[col] = pd.to_datetime(deliveries[col], errors='coerce')

print(deliveries[date_cols].dtypes)
print(deliveries[date_cols].isnull().sum())

Pickup_Date             datetime64[ns]
Actual_Delivery_Date    datetime64[ns]
dtype: object
Pickup_Date             1766
Actual_Delivery_Date    1963
dtype: int64


In [94]:
#Clean Expected_Days

deliveries['Expected_Days'] = pd.to_numeric(
    deliveries['Expected_Days'], errors='coerce'
)

# Valid range 1-14 days
print(f"Expected_Days < 1: {(deliveries['Expected_Days'] < 1).sum()}")
print(f"Expected_Days > 14: {(deliveries['Expected_Days'] > 14).sum()}")

deliveries['Expected_Days'] = deliveries['Expected_Days'].where(
    (deliveries['Expected_Days'] >= 1) &
    (deliveries['Expected_Days'] <= 14),
    np.nan
)

# Fill nulls with median
deliveries['Expected_Days'] = deliveries['Expected_Days'].fillna(
    deliveries['Expected_Days'].median()
).round(0).astype(int)

print(deliveries['Expected_Days'].describe())

Expected_Days < 1: 0
Expected_Days > 14: 0
count    9791.000000
mean        5.013584
std         1.418584
min         3.000000
25%         4.000000
50%         5.000000
75%         6.000000
max         7.000000
Name: Expected_Days, dtype: float64


In [95]:
# Clean Actual_Delivery_Days

deliveries['Actual_Delivery_Days'] = pd.to_numeric(
    deliveries['Actual_Delivery_Days'], errors='coerce'
)

# Valid range 1-30 days
print(f"Null Actual_Days: {deliveries['Actual_Delivery_Days'].isnull().sum()}")
print(f"Negative Actual_Days: {(deliveries['Actual_Delivery_Days'] < 1).sum()}")
print(f"Extreme Actual_Days (>30): {(deliveries['Actual_Delivery_Days'] > 30).sum()}")

# Fix invalid values
deliveries['Actual_Delivery_Days'] = deliveries['Actual_Delivery_Days'].where(
    (deliveries['Actual_Delivery_Days'] >= 1) &
    (deliveries['Actual_Delivery_Days'] <= 30),
    np.nan
)

# Fill nulls with Expected_Days + partner average delay
partner_avg_delay = {
    'Delhivery': 1, 'Blue Dart': 1, 'DTDC': 4,
    'Ekart': 2, 'XpressBees': 3, 'Amazon Logistics': 1, 'Unknown': 2
}

def fill_actual_days(row):
    if pd.notnull(row['Actual_Delivery_Days']):
        return row['Actual_Delivery_Days']
    avg_delay = partner_avg_delay.get(row['Delivery_Partner'], 2)
    return row['Expected_Days'] + avg_delay

deliveries['Actual_Delivery_Days'] = deliveries.apply(
    fill_actual_days, axis=1
).round(0).astype(int)

print(deliveries['Actual_Delivery_Days'].describe())

Null Actual_Days: 1658
Negative Actual_Days: 0
Extreme Actual_Days (>30): 0
count    9791.000000
mean        6.560004
std         2.288053
min         1.000000
25%         5.000000
50%         6.000000
75%         8.000000
max        14.000000
Name: Actual_Delivery_Days, dtype: float64


In [96]:
#Clean and Recalculate Delay_Days

# First check what we have
print(f"Null Delay_Days: {deliveries['Delay_Days'].isnull().sum()}")

deliveries['Delay_Days'] = pd.to_numeric(
    deliveries['Delay_Days'], errors='coerce'
)

# Recalculate from clean Actual and Expected
# This fixes both null and incorrect values
deliveries['Delay_Days'] = (
    deliveries['Actual_Delivery_Days'] - deliveries['Expected_Days']
)

# Classify delay type
# Negative = early delivery (valid)
# Extreme negative (< -5) = likely data error
extreme_negative = (deliveries['Delay_Days'] < -5).sum()
print(f"Extreme negative delays (< -5): {extreme_negative}")

# Cap extreme negatives — keep genuine early deliveries
deliveries['Delay_Days'] = deliveries['Delay_Days'].clip(lower=-3)

print(deliveries['Delay_Days'].describe())
print(f"\nDelay distribution:")
print(deliveries['Delay_Days'].value_counts().sort_index().head(15))

Null Delay_Days: 1142
Extreme negative delays (< -5): 0
count    9791.000000
mean        1.546420
std         1.801788
min        -2.000000
25%         0.000000
50%         1.000000
75%         2.000000
max         7.000000
Name: Delay_Days, dtype: float64

Delay distribution:
Delay_Days
-2     147
-1     981
 0    1579
 1    2591
 2    2128
 3    1123
 4     556
 5     323
 6     180
 7     183
Name: count, dtype: int64


In [97]:
#Standardize Delivery_Status

def standardize_status(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    if val in ['delivered','deliverd','complete','completed']:
        return 'Delivered'
    if 'out for' in val or val == 'ofd':
        return 'Out for Delivery'
    if 'transit' in val or 'intransit' in val:
        return 'In Transit'
    if 'fail' in val:
        return 'Failed'
    if 'rto' in val or 'returned to origin' in val or 'return to origin' in val:
        return 'Returned to Origin'
    if 'delay' in val or 'behind' in val:
        return 'Delayed'
    return np.nan

deliveries['Delivery_Status'] = deliveries['Delivery_Status'].apply(
    standardize_status
)

# Fill nulls — derive from Delay_Days
def derive_status(row):
    if pd.notnull(row['Delivery_Status']):
        return row['Delivery_Status']
    delay = row['Delay_Days']
    if delay <= 0:    return 'Delivered'
    elif delay <= 2:  return 'In Transit'
    elif delay <= 5:  return 'Delayed'
    else:             return 'Severely Delayed'

deliveries['Delivery_Status'] = deliveries.apply(derive_status, axis=1)

print(deliveries['Delivery_Status'].value_counts())

Delivery_Status
Delivered           6172
In Transit          1576
Delayed             1243
Failed               511
Out for Delivery     289
Name: count, dtype: int64


In [98]:
#Clean Delivery_Attempts

deliveries['Delivery_Attempts'] = pd.to_numeric(
    deliveries['Delivery_Attempts'], errors='coerce'
)

# Valid range 1-5
print(f"Invalid attempts (<1): {(deliveries['Delivery_Attempts'] < 1).sum()}")
print(f"Invalid attempts (>5): {(deliveries['Delivery_Attempts'] > 5).sum()}")

deliveries['Delivery_Attempts'] = deliveries['Delivery_Attempts'].where(
    (deliveries['Delivery_Attempts'] >= 1) &
    (deliveries['Delivery_Attempts'] <= 5),
    np.nan
)

# Failed deliveries should have at least 1 attempt
deliveries.loc[
    (deliveries['Delivery_Status'] == 'Failed') &
    (deliveries['Delivery_Attempts'].isnull()),
    'Delivery_Attempts'
] = 1

# Fill remaining nulls with 1
deliveries['Delivery_Attempts'] = deliveries['Delivery_Attempts'].fillna(1).astype(int)

print(deliveries['Delivery_Attempts'].value_counts())

Invalid attempts (<1): 0
Invalid attempts (>5): 0
Delivery_Attempts
1    8539
2     820
3     432
Name: count, dtype: int64


In [99]:
# Clean Delivery_Cost

def clean_price(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip()
    val = re.sub(r'[₹Rs\.INR,/\-\s]', '', val)
    try:
        result = float(val)
        return result if result > 0 else np.nan
    except:
        return np.nan

deliveries['Delivery_Cost'] = deliveries['Delivery_Cost'].apply(clean_price)

# Negative delivery cost — data error — take absolute
neg_cost = (deliveries['Delivery_Cost'] < 0).sum()
print(f"Negative delivery costs: {neg_cost}")
deliveries['Delivery_Cost'] = deliveries['Delivery_Cost'].abs()

# Fill nulls with partner median
deliveries['Delivery_Cost'] = deliveries.groupby('Delivery_Partner')['Delivery_Cost'].transform(
    lambda x: x.fillna(x.median())
)

print(deliveries['Delivery_Cost'].describe())

Negative delivery costs: 0
count     9791.000000
mean      9156.236237
std       3678.741348
min        353.000000
25%       7177.500000
50%       9129.000000
75%      11221.000000
max      23578.000000
Name: Delivery_Cost, dtype: float64


In [100]:
# Clean Customer_Rating

deliveries['Customer_Rating'] = pd.to_numeric(
    deliveries['Customer_Rating'], errors='coerce'
)

# Valid range 1-5
print(f"Rating > 5: {(deliveries['Customer_Rating'] > 5).sum()}")
print(f"Rating < 1: {(deliveries['Customer_Rating'] < 1).sum()}")

deliveries['Customer_Rating'] = deliveries['Customer_Rating'].where(
    (deliveries['Customer_Rating'] >= 1) &
    (deliveries['Customer_Rating'] <= 5),
    np.nan
)

# Fill nulls based on delivery status — logic-based imputation
status_rating_map = {
    'Delivered': 4.0,
    'Out for Delivery': 3.5,
    'In Transit': 3.0,
    'Delayed': 2.5,
    'Failed': 1.5,
    'Returned to Origin': 2.0
}

deliveries['Customer_Rating'] = deliveries.apply(
    lambda row: status_rating_map.get(row['Delivery_Status'], 3.0)
    if pd.isnull(row['Customer_Rating'])
    else row['Customer_Rating'],
    axis=1
)

print(deliveries['Customer_Rating'].describe())

Rating > 5: 932
Rating < 1: 0
count    9791.000000
mean        3.543765
std         1.244210
min         1.000000
25%         3.000000
50%         4.000000
75%         4.000000
max         5.000000
Name: Customer_Rating, dtype: float64


In [101]:
# Clean Failure_Reason

# Standardize missing encodings
deliveries['Failure_Reason'] = deliveries['Failure_Reason'].replace(
    ['nan','NaN','NA','N/A','None','','null','-'], np.nan
)

# Failure reason only valid for Failed status
non_failed_with_reason = (
    (deliveries['Delivery_Status'] != 'Failed') &
    deliveries['Failure_Reason'].notna()
).sum()
print(f"Non-failed with failure reason: {non_failed_with_reason}")

# Clear for non-failed
deliveries.loc[
    deliveries['Delivery_Status'] != 'Failed',
    'Failure_Reason'
] = np.nan

# Fill missing for failed
deliveries.loc[
    (deliveries['Delivery_Status'] == 'Failed') &
    (deliveries['Failure_Reason'].isnull()),
    'Failure_Reason'
] = 'Reason Not Captured'

print(deliveries['Failure_Reason'].value_counts())

Non-failed with failure reason: 0
Failure_Reason
Reason Not Captured          140
Restricted Access             70
Customer Not Available        69
Address Not Found             66
Wrong Address                 65
Customer Refused Delivery     60
Natural Disaster              41
Name: count, dtype: int64


In [102]:
#Outlier Treatment
def iqr_cap(series, col_name):
    Q1  = series.quantile(0.25)
    Q3  = series.quantile(0.75)
    IQR = Q3 - Q1
    lower  = Q1 - 1.5 * IQR
    upper  = Q3 + 1.5 * IQR
    out    = ((series < lower) | (series > upper)).sum()
    print(f"  {col_name}: {out} outliers | bounds [{lower:.2f}, {upper:.2f}]")
    return series.clip(lower=lower, upper=upper)

print("\nOutlier treatment:")
deliveries['Delivery_Cost']        = iqr_cap(deliveries['Delivery_Cost'], 'Delivery_Cost')
deliveries['Actual_Delivery_Days'] = iqr_cap(deliveries['Actual_Delivery_Days'], 'Actual_Days')
deliveries['Delay_Days']           = iqr_cap(deliveries['Delay_Days'], 'Delay_Days')


Outlier treatment:
  Delivery_Cost: 672 outliers | bounds [1112.25, 17286.25]
  Actual_Days: 113 outliers | bounds [0.50, 12.50]
  Delay_Days: 363 outliers | bounds [-3.00, 5.00]


In [103]:
#Feature Engineering

today = pd.Timestamp.now()

# 1. Delay Flag
deliveries['Delay_Flag'] = (deliveries['Delay_Days'] > 0).astype(int)

# 2. On Time Flag
deliveries['On_Time_Flag'] = (deliveries['Delay_Days'] <= 0).astype(int)

# 3. Early Delivery Flag
deliveries['Early_Delivery_Flag'] = (deliveries['Delay_Days'] < 0).astype(int)

# 4. Is Failed Delivery
deliveries['Is_Failed'] = (
    deliveries['Delivery_Status'] == 'Failed'
).astype(int)

# 5. Multiple Attempt Flag
deliveries['Multiple_Attempt_Flag'] = (
    deliveries['Delivery_Attempts'] > 1
).astype(int)

# 6. Delivery Speed Band
def speed_band(days):
    if pd.isnull(days): return 'Unknown'
    if days <= 2:   return 'Express (1-2 days)'
    elif days <= 4: return 'Standard (3-4 days)'
    elif days <= 7: return 'Slow (5-7 days)'
    else:           return 'Very Slow (8+ days)'

deliveries['Delivery_Speed_Band'] = deliveries['Actual_Delivery_Days'].apply(
    speed_band
)

# 7. Cost Per Day
deliveries['Cost_Per_Day'] = round(
    deliveries['Delivery_Cost'] / deliveries['Actual_Delivery_Days'], 2
)

# 8. Delay Severity
def delay_severity(days):
    if pd.isnull(days): return 'Unknown'
    if days < 0:    return 'Early'
    elif days == 0: return 'On Time'
    elif days <= 2: return 'Slightly Delayed'
    elif days <= 5: return 'Moderately Delayed'
    else:           return 'Severely Delayed'

deliveries['Delay_Severity'] = deliveries['Delay_Days'].apply(delay_severity)

# 9. Partner Reliability Score
# Average on-time rate per partner
partner_ontime = deliveries.groupby('Delivery_Partner')['On_Time_Flag'].transform('mean')
deliveries['Partner_Reliability_Score'] = round(partner_ontime * 100, 2)

# 10. Date Features from Pickup_Date
deliveries['Pickup_Month']   = deliveries['Pickup_Date'].dt.month
deliveries['Pickup_Quarter'] = deliveries['Pickup_Date'].dt.quarter
deliveries['Pickup_DayOfWeek'] = deliveries['Pickup_Date'].dt.day_name()
deliveries['Is_Weekend_Pickup'] = deliveries['Pickup_Date'].dt.dayofweek.isin([5,6]).astype(int)

# 11. Is Festive Season Delivery
deliveries['Is_Festive_Season'] = deliveries['Pickup_Month'].isin([10,11,12]).astype(int)

# 12. Low Rating Flag
deliveries['Low_Rating_Flag'] = (deliveries['Customer_Rating'] <= 2).astype(int)

print(deliveries[['Delay_Flag','On_Time_Flag','Is_Failed',
                   'Delivery_Speed_Band','Delay_Severity',
                   'Partner_Reliability_Score']].head(10))

   Delay_Flag  On_Time_Flag  Is_Failed  Delivery_Speed_Band  \
0           1             0          0      Slow (5-7 days)   
1           0             1          0   Express (1-2 days)   
2           0             1          0  Standard (3-4 days)   
3           1             0          0  Standard (3-4 days)   
4           1             0          1  Very Slow (8+ days)   
5           1             0          0  Very Slow (8+ days)   
6           1             0          0      Slow (5-7 days)   
7           0             1          0      Slow (5-7 days)   
8           1             0          0      Slow (5-7 days)   
9           1             0          0  Very Slow (8+ days)   

       Delay_Severity  Partner_Reliability_Score  
0    Slightly Delayed                      40.37  
1               Early                      34.18  
2             On Time                      23.22  
3    Slightly Delayed                      40.37  
4  Moderately Delayed                       0.00  


In [104]:
# Normalization

from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Min-Max Normalization
scaler_mm = MinMaxScaler()
deliveries['Delivery_Cost_Normalized'] = scaler_mm.fit_transform(
    deliveries[['Delivery_Cost']]
)
deliveries['Rating_Normalized'] = scaler_mm.fit_transform(
    deliveries[['Customer_Rating']]
)
deliveries['Delay_Days_Normalized'] = scaler_mm.fit_transform(
    deliveries[['Delay_Days']]
)

# Z-Score Standardization
scaler_z = StandardScaler()
deliveries['Cost_Zscore'] = scaler_z.fit_transform(
    deliveries[['Delivery_Cost']]
)
deliveries['Delay_Zscore'] = scaler_z.fit_transform(
    deliveries[['Delay_Days']]
)

# Log Transformation
deliveries['Cost_Log']         = np.log1p(deliveries['Delivery_Cost'])
deliveries['Actual_Days_Log']  = np.log1p(deliveries['Actual_Delivery_Days'])

print(deliveries[['Delivery_Cost_Normalized','Rating_Normalized',
                   'Delay_Zscore','Cost_Log']].describe())


       Delivery_Cost_Normalized  Rating_Normalized  Delay_Zscore     Cost_Log
count               9791.000000        9791.000000  9.791000e+03  9791.000000
mean                   0.495299           0.635941  1.306278e-17     8.994611
std                    0.217665           0.311053  1.000051e+00     0.599233
min                    0.000000           0.000000 -2.099351e+00     7.015039
25%                    0.375000           0.500000 -8.965104e-01     8.878846
50%                    0.495657           0.750000 -2.950898e-01     9.119321
75%                    0.625000           0.750000  3.063307e-01     9.325631
max                    1.000000           1.000000  2.110592e+00     9.757725


In [105]:
# Encoding Categorical Variables

# 1. Label Encoding — Delivery_Status (ordinal)
status_order = {
    'Failed':0, 'Returned to Origin':1, 'Severely Delayed':2,
    'Delayed':3, 'In Transit':4, 'Out for Delivery':5, 'Delivered':6
}
deliveries['Status_Encoded'] = deliveries['Delivery_Status'].map(status_order)

# 2. Label Encoding — Delay_Severity (ordinal)
severity_order = {
    'Severely Delayed':0, 'Moderately Delayed':1,
    'Slightly Delayed':2, 'On Time':3, 'Early':4, 'Unknown':-1
}
deliveries['Severity_Encoded'] = deliveries['Delay_Severity'].map(severity_order)

# 3. Label Encoding — Delivery_Speed_Band (ordinal)
speed_order = {
    'Very Slow (8+ days)':0, 'Slow (5-7 days)':1,
    'Standard (3-4 days)':2, 'Express (1-2 days)':3, 'Unknown':-1
}
deliveries['Speed_Encoded'] = deliveries['Delivery_Speed_Band'].map(speed_order)

# 4. One Hot Encoding — Delivery_Partner
partner_dummies = pd.get_dummies(
    deliveries['Delivery_Partner'],
    prefix='Partner',
    drop_first=True
)
deliveries = pd.concat([deliveries, partner_dummies], axis=1)

# 5. Cyclical Encoding — Pickup_Month
deliveries['Month_Sin'] = np.sin(2 * np.pi * deliveries['Pickup_Month'] / 12)
deliveries['Month_Cos'] = np.cos(2 * np.pi * deliveries['Pickup_Month'] / 12)

print(f"Shape after encoding: {deliveries.shape}")
print(deliveries[['Status_Encoded','Severity_Encoded','Speed_Encoded']].head())

Shape after encoding: (9791, 47)
   Status_Encoded  Severity_Encoded  Speed_Encoded
0               6                 2              1
1               6                 4              3
2               6                 3              2
3               6                 2              2
4               0                 1              0


In [106]:
# Join with Orders to validate business logic
try:
    orders = pd.read_csv('KartZone_Orders_Final.csv')

    merged = deliveries.merge(
        orders[['Order_ID','Order_Status','Category']],
        on='Order_ID',
        how='left'
    )

    # Check: Returned orders should have delayed or failed delivery
    returned_on_time = (
        (merged['Order_Status'] == 'Returned') &
        (merged['On_Time_Flag'] == 1)
    ).sum()
    print(f"Returned orders delivered on time: {returned_on_time}")

    # Check: Delivery partner performance by category
    print("\nDelay rate by partner:")
    print(deliveries.groupby('Delivery_Partner')['Delay_Flag'].mean().sort_values(ascending=False).round(3))

    print("\nAverage rating by partner:")
    print(deliveries.groupby('Delivery_Partner')['Customer_Rating'].mean().sort_values(ascending=False).round(2))

except:
    print("Orders table not available for cross validation")

Returned orders delivered on time: 380

Delay rate by partner:
Delivery_Partner
DTDC                1.000
XpressBees          0.847
Unknown             0.778
Blue Dart           0.768
Ekart               0.658
Delhivery           0.596
Amazon Logistics    0.518
Name: Delay_Flag, dtype: float64

Average rating by partner:
Delivery_Partner
Amazon Logistics    4.02
Delhivery           3.95
Ekart               3.71
Blue Dart           3.61
Unknown             3.53
XpressBees          3.02
DTDC                2.61
Name: Customer_Rating, dtype: float64


In [107]:
print("\n" + "="*55)
print("  DELIVERIES — FINAL DATA QUALITY REPORT")
print("="*55)
print(f"  Total rows              : {len(deliveries):,}")
print(f"  Total columns           : {len(deliveries.columns)}")
print(f"  Remaining nulls         : {deliveries.isnull().sum().sum():,}")
print(f"  Duplicates              : {deliveries.duplicated().sum()}")

print(f"\n  DELIVERY PERFORMANCE METRICS:")
ontime  = deliveries['On_Time_Flag'].mean() * 100
delayed = deliveries['Delay_Flag'].mean() * 100
failed  = deliveries['Is_Failed'].mean() * 100
early   = deliveries['Early_Delivery_Flag'].mean() * 100

print(f"  On-Time Rate            : {ontime:.2f}%")
print(f"  Delayed Rate            : {delayed:.2f}%")
print(f"  Failed Delivery Rate    : {failed:.2f}%")
print(f"  Early Delivery Rate     : {early:.2f}%")
print(f"  Avg Customer Rating     : {deliveries['Customer_Rating'].mean():.2f}")
print(f"  Avg Delay Days          : {deliveries['Delay_Days'].mean():.2f}")
print(f"  Avg Delivery Cost       : ₹{deliveries['Delivery_Cost'].mean():.2f}")

print(f"\n  PARTNER PERFORMANCE:")
print(deliveries.groupby('Delivery_Partner').agg(
    Orders=('Order_ID','count'),
    OnTime_Rate=('On_Time_Flag','mean'),
    Avg_Rating=('Customer_Rating','mean'),
    Avg_Delay=('Delay_Days','mean'),
    Avg_Cost=('Delivery_Cost','mean')
).round(2).sort_values('OnTime_Rate', ascending=False))

print(f"\n  DELIVERY STATUS:")
print(deliveries['Delivery_Status'].value_counts())

print(f"\n  DELAY SEVERITY:")
print(deliveries['Delay_Severity'].value_counts())


  DELIVERIES — FINAL DATA QUALITY REPORT
  Total rows              : 9,791
  Total columns           : 47
  Remaining nulls         : 21,839
  Duplicates              : 0

  DELIVERY PERFORMANCE METRICS:
  On-Time Rate            : 27.65%
  Delayed Rate            : 72.35%
  Failed Delivery Rate    : 5.22%
  Early Delivery Rate     : 11.52%
  Avg Customer Rating     : 3.54
  Avg Delay Days          : 1.49
  Avg Delivery Cost       : ₹9123.22

  PARTNER PERFORMANCE:
                  Orders  OnTime_Rate  Avg_Rating  Avg_Delay  Avg_Cost
Delivery_Partner                                                      
Amazon Logistics     825         0.48        4.02       0.21   9032.93
Delhivery           2554         0.40        3.95       0.62   9430.80
Ekart               1855         0.34        3.71       1.14   8455.72
Blue Dart           1723         0.23        3.61       1.34  12180.25
Unknown              459         0.22        3.53       1.58   8841.78
XpressBees           927        

In [108]:
# Save full cleaned version
deliveries.to_csv('KartZone_Deliveries_Clean.csv', index=False)
print("Saved → KartZone_Deliveries_Clean.csv ✓")

# Essential columns for SQL load
essential_cols = [
    'Delivery_ID','Order_ID','Delivery_Partner','Warehouse_City',
    'Pickup_Date','Expected_Days','Actual_Delivery_Days',
    'Delay_Days','Actual_Delivery_Date','Delivery_Status',
    'Delivery_Attempts','Failure_Reason','Delivery_Cost',
    'Customer_Rating',
    'Delay_Flag','On_Time_Flag','Early_Delivery_Flag',
    'Is_Failed','Multiple_Attempt_Flag',
    'Delivery_Speed_Band','Cost_Per_Day','Delay_Severity',
    'Partner_Reliability_Score',
    'Pickup_Month','Pickup_Quarter','Is_Weekend_Pickup',
    'Is_Festive_Season','Low_Rating_Flag'
]
deliveries[essential_cols].to_csv('KartZone_Deliveries_Final.csv', index=False)
print("Saved → KartZone_Deliveries_Final.csv ✓")

Saved → KartZone_Deliveries_Clean.csv ✓
Saved → KartZone_Deliveries_Final.csv ✓


In [ ]:
#Load clean data to SQL Server

In [1]:
pip install pyodbc sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [2]:
#Create SQL connection
from sqlalchemy import create_engine
import urllib

In [3]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-O4D9R9M;"
    "DATABASE=Projectdtbase;"
    "Trusted_Connection=yes;"
)

In [4]:
engine = create_engine("mssql+pyodbc:///?odbc_connect=%s" % params)

In [7]:
# Load all 4 final files
import pandas as pd
customers  = pd.read_csv('KartZone_Customers_Final.csv')
products   = pd.read_csv('KartZone_Products_Final.csv')
orders     = pd.read_csv('KartZone_Orders_Final.csv')
deliveries = pd.read_csv('KartZone_Deliveries_Final.csv')

# Push to SQL Server
customers.to_sql('Customers',   con=engine, if_exists='replace', index=False)
products.to_sql('Products',     con=engine, if_exists='replace', index=False)
orders.to_sql('Orders',         con=engine, if_exists='replace', index=False)
deliveries.to_sql('Deliveries', con=engine, if_exists='replace', index=False)

print("All 4 tables loaded successfully!")

C:\Users\admin\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


All 4 tables loaded successfully!


In [ ]:
#see i could transferred the data but when i run above query it gives 
"""
# Load all 4 final files
import pandas as pd
customers  = pd.read_csv('KartZone_Customers_Final.csv')
products   = pd.read_csv('KartZone_Products_Final.csv')
orders     = pd.read_csv('KartZone_Orders_Final.csv')
deliveries = pd.read_csv('KartZone_Deliveries_Final.csv')

# Push to SQL Server
customers.to_sql('Customers',   con=engine, if_exists='replace', index=False)
products.to_sql('Products',     con=engine, if_exists='replace', index=False)
orders.to_sql('Orders',         con=engine, if_exists='replace', index=False)
deliveries.to_sql('Deliveries', con=engine, if_exists='replace', index=False)

print("All 4 tables loaded successfully!")
C:\Users\admin\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())
All 4 tables loaded successfully!""" what is this error